# Complete Norwegian field and area development with NeqSim

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fielddevelopment/norwegian_field_and_area_development_with_neqsim.ipynb)

> **FOR EDUCATION ONLY — NOT A PROJECT DECISION BASIS.** This notebook uses a fictitious discovery called **Aurora Sør** in a real Norwegian Continental Shelf setting. Public SODIR and Gassco information is used only for geographic, infrastructure and transport-system context. Reservoir, well, host-capacity, process, cost, tariff and commercial inputs are synthetic unless a cell explicitly identifies an official public observation.

This is a complete concept-selection teaching study: PVT, reservoir, wells, SURF, subsea maps, tie-in distances, gas-pipeline topology, detailed processing, gas injection, product specifications, annual bottlenecks, holdback, host utilization, modifications, emissions, NPV, IRR and break-even.

## How to use this notebook

This notebook is intentionally long and follows the sequence of an early-phase field-development study. Run it from top to bottom. The integrated NeqSim portfolio calculation is deliberately concentrated in one section so mutable reservoir and process models are never reused incorrectly.

The study compares a standalone FPSO with gas injection, natural depletion, a direct host-priority tieback, a managed-capacity tieback and a more remote host route. It then tests how a real existing ProcessSystem, a multi-area ProcessModel and an existing shared SURF system would replace the teaching models.

Public coordinates and pipeline geometries are screening data. They are not survey-quality coordinates, navigation data, ownership evidence, available capacity, consent to tie in, or proof of technical/commercial access.

## Study questions

1. Is a standalone development or a brownfield tieback most attractive for the fictitious discovery?

2. Which reservoir, well, SURF, facility and export constraints shape production?

3. How much production is held back or deferred by a producing host?

4. When does shared equipment become a bottleneck, and what modification should be studied?

5. How do product specifications, gas transport capacity, tariffs, emissions and uncertainty affect the ranking?

6. How can public NCS geometry be used responsibly to build a candidate-host and route long list?

## Learning objectives

By the end, you should be able to assemble an auditable evidence/assumption ledger, inspect a compositional fluid, read a NeqSim process topology, calculate lifecycle profiles, distinguish requested from admitted capacity utilization, compare area routes and reproduce the economic ranking.

You should also be able to replace the teaching flowsheets with an operator-owned detailed plant and SURF model without changing the surrounding lifecycle workflow.

## Table of contents

1. Environment and model provenance

2. Official SODIR and Gassco context

3. Geospatial NCS screening and subsea topology

4. Fictitious discovery and decision basis

5. Compositional PVT

6. Reservoir, wells and injection

7. SURF and gas-pipeline hydraulics

8. Detailed greenfield facility

9. Integrated lifecycle results

10. Brownfield hosts, holdback and existing infrastructure

11. Area development, economics and uncertainty

12. Detailed well, SURF and process engineering
13. Validation, limitations and conclusions

## Method reference

The workflow implements the reservoir-to-market sequence described in the NeqSim Field Lifecycle Simulation guide: compositional PVT and material balance; well/SURF deliverability; a detailed ProcessSystem or ProcessModel; gas allocation and injection; annual capacity and product checks; then economics and ranking.

Read the current guide alongside the notebook: https://equinor.github.io/neqsim/fielddevelopment/FIELD_LIFECYCLE_SIMULATION.html

## 1. Environment

The notebook currently builds the NeqSim feature branch containing the lifecycle API. After that API is released, replace the source build with the standard released package. A source build makes the educational notebook reproducible before publication.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

NEQSIM_ROOT = Path("/content/neqsim-src")
NEQSIM_BRANCH = "agent/fix-lifecycle-power-aggregation"

if not (NEQSIM_ROOT / ".git").exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", NEQSIM_BRANCH,
        "https://github.com/equinor/neqsim.git", str(NEQSIM_ROOT)
    ], check=True)

subprocess.run([
    str(NEQSIM_ROOT / "mvnw"), "-B", "-ntp", "-DskipTests",
    "-Dmaven.javadoc.skip=true", "package"
], cwd=NEQSIM_ROOT, check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "JPype1", "pandas", "numpy", "matplotlib", "seaborn", "scipy",
    "requests", "geopandas", "shapely", "pyproj", "folium", "networkx"
], check=True)

print("NeqSim source and Python dependencies are ready.")

## JVM lifecycle

JPype starts one Java virtual machine per Python kernel. If you rebuild NeqSim after starting the JVM, restart the Colab runtime before loading the new classes.

In [ ]:
sys.path.insert(0, str(NEQSIM_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init

ns = neqsim_init(str(NEQSIM_ROOT), verbose=False)
print("JVM started:", ns.PROJECT_ROOT)

## Python analysis stack

GeoPandas/Shapely/pyproj handle geometry and distances, Folium creates an interactive screening map, NetworkX represents route topology, and pandas/matplotlib/seaborn handle results. NeqSim remains the source for thermodynamics, flow, process, capacity and lifecycle calculations.

In [ ]:
import json
import math
import warnings
from datetime import date

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import HTML, Markdown, display
from pyproj import Geod
from scipy.optimize import brentq
from shapely.geometry import LineString, MultiLineString, Point, Polygon
from shapely.ops import unary_union

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
TODAY = date.today().isoformat()
print("Analysis date:", TODAY)

## Load NeqSim classes

The imports below cover the complete study chain. Java classes are loaded explicitly, which makes it easy to see which calculations are performed by NeqSim.

In [ ]:
from jpype import JClass

NorwegianOilFieldCase = JClass("neqsim.process.fielddevelopment.lifecycle.NorwegianOilFieldCase")
FieldLifecycleEvaluator = JClass("neqsim.process.fielddevelopment.lifecycle.FieldLifecycleEvaluator")
AreaDevelopmentEvaluator = JClass("neqsim.process.fielddevelopment.lifecycle.AreaDevelopmentEvaluator")
FacilityModificationPlanner = JClass("neqsim.process.fielddevelopment.lifecycle.FacilityModificationPlanner")
CashFlowEngine = JClass("neqsim.process.fielddevelopment.economics.CashFlowEngine")
ThermodynamicOperations = JClass("neqsim.thermodynamicoperations.ThermodynamicOperations")
Stream = JClass("neqsim.process.equipment.stream.Stream")
PipeBeggsAndBrills = JClass("neqsim.process.equipment.pipeline.PipeBeggsAndBrills")
InjectionWellModel = JClass("neqsim.process.fielddevelopment.reservoir.InjectionWellModel")
InjectionType = JClass("neqsim.process.fielddevelopment.reservoir.InjectionWellModel$InjectionType")
WellCostEstimator = JClass("neqsim.process.mechanicaldesign.subsea.WellCostEstimator")
SURFCostEstimator = JClass("neqsim.process.mechanicaldesign.subsea.SURFCostEstimator")
SubseaRegion = JClass("neqsim.process.mechanicaldesign.subsea.SubseaCostEstimator$Region")
ProcessModel = JClass("neqsim.process.processmodel.ProcessModel")
FieldLifecycleModel = JClass("neqsim.process.fielddevelopment.lifecycle.FieldLifecycleModel")

print("Lifecycle API loaded.")

## Reproducibility controls

Random numbers are used only for educational uncertainty propagation. A fixed seed makes all stochastic tables and figures repeatable.

In [ ]:
RANDOM_SEED = 4230
rng = np.random.default_rng(RANDOM_SEED)
BARREL_PER_SM3 = 6.28981077
DAYS_PER_YEAR = 365.25
UTM_NORTH_SEA = "EPSG:25831"

study_manifest = {
    "notebook_version": "2.0",
    "case": "Aurora Sør (fictitious)",
    "random_seed": RANDOM_SEED,
    "neqsim_branch": NEQSIM_BRANCH,
    "public_data_access_date": TODAY,
}
display(pd.Series(study_manifest, name="value").to_frame())

## 2. Source hierarchy and provenance

Every input belongs to one of four classes: official public observation, derived public-context metric, synthetic engineering assumption, or calculated result. A real project should add document identifiers, owners, review status, validity dates and uncertainty.

In [ ]:
provenance = pd.DataFrame([
    ["SODIR fields/discoveries/facilities/pipelines", "official public observation", "SODIR REST/FactPages", "context and screening only"],
    ["Gassco pipeline dimensions/routes/areas", "official public observation", "Gassco transport map", "transport context only"],
    ["Gassco booking concepts", "official public observation", "Gassco shipper pages", "commercial constraint logic"],
    ["Aurora Sør coordinates and reservoir", "synthetic assumption", "this notebook", "teaching case"],
    ["Host A/B capacity and production", "synthetic assumption", "NeqSim factory", "teaching comparison"],
    ["PVT/process/lifecycle outputs", "calculated result", "NeqSim", "case-specific"],
    ["NPV/break-even/sensitivities", "calculated result", "NeqSim plus stated scenarios", "education only"],
], columns=["item", "classification", "source", "permitted_use"])
display(provenance)

## Official SODIR sources

SODIR states that FactPages and downloadable/REST datasets are openly available. The directorate also cautions that map data have varying positional accuracy and are not for navigation. We therefore use geometry only for early screening.

Open data: https://www.sodir.no/en/facts/data-and-analyses/open-data/

FactPages: https://factpages.sodir.no/en

Resource Accounts 2025: https://www.sodir.no/en/whats-new/publications/reports/resource-accounts/resource-accounts-2025/

## Official Gassco sources

Gassco publishes the transport-system map, tariff-area overview, standard agreement structure and booking concepts. The public booking page distinguishes primary-market periods, secondary-market trading, nominations and rebooking. Access to actual booking and reporting systems is restricted to authorized users.

Transport map: https://gassco.eu/en/transport-map/

Tariffs and areas: https://gassco.eu/en/shippers/tariffs-and-areas/

Transport agreements: https://gassco.eu/en/shippers/transport-agreements/

Capacity booking/reporting: https://gassco.eu/en/shippers/capacity-booking-and-reporting/

## What public sources do not provide

Public pages do not establish ullage at a host, tie-in consent, shutdown windows, measurement uncertainty, equipment condition, contractual gas quality at a selected delivery point, project tariffs, ownership alignment, or the right to transport. Those must remain explicit gates.

The notebook never infers available capacity from a public capacity number or a declining field profile.

## SODIR DataService layer map

The official DataService mapping identifies field layer 7100, field reserves 7113, discovery 7000, facility 6000 and pipeline 6100. The reader below uses ArcGIS REST JSON and preserves the returned attributes.

In [ ]:
SODIR_DATA = "https://factmaps.sodir.no/api/rest/services/DataService/Data/FeatureServer"
SODIR_LAYERS = {
    "facility": 6000,
    "pipeline": 6100,
    "discovery": 7000,
    "field": 7100,
    "field_reserves": 7113,
}

def sodir_features(layer_id, where="1=1", out_fields="*", return_geometry=True, count=2000):
    params = {
        "f": "json",
        "where": where,
        "outFields": out_fields,
        "returnGeometry": str(return_geometry).lower(),
        "outSR": 4326,
        "resultRecordCount": count,
    }
    response = requests.get(f"{SODIR_DATA}/{layer_id}/query", params=params, timeout=90)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(payload["error"])
    return payload.get("features", [])

## Convert ArcGIS geometry

SODIR returns Esri point, path and ring geometries. The conversion is adequate for screening maps and distance ranking. It does not repair topology or replace GIS QA.

In [ ]:
def esri_to_shape(geometry):
    if not geometry:
        return None
    if "x" in geometry and "y" in geometry:
        return Point(float(geometry["x"]), float(geometry["y"]))
    if "paths" in geometry:
        lines = [LineString(path) for path in geometry["paths"] if len(path) >= 2]
        if not lines:
            return None
        return lines[0] if len(lines) == 1 else MultiLineString(lines)
    if "rings" in geometry:
        rings = [Polygon(ring) for ring in geometry["rings"] if len(ring) >= 4]
        return unary_union(rings) if rings else None
    return None

def features_to_gdf(features):
    rows = []
    for feature in features:
        row = dict(feature.get("attributes", {}))
        row["geometry"] = esri_to_shape(feature.get("geometry"))
        rows.append(row)
    if not rows:
        return gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs="EPSG:4326")
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")

## Download the public NCS snapshot

The cell fails gracefully if the public service is unavailable. The synthetic NeqSim analysis remains executable, but mapping cells will report that no public geometry was loaded.

In [ ]:
try:
    field_gdf = features_to_gdf(sodir_features(SODIR_LAYERS["field"]))
    discovery_gdf = features_to_gdf(sodir_features(SODIR_LAYERS["discovery"]))
    facility_gdf = features_to_gdf(sodir_features(SODIR_LAYERS["facility"]))
    pipeline_gdf = features_to_gdf(sodir_features(SODIR_LAYERS["pipeline"]))
    reserve_features = sodir_features(SODIR_LAYERS["field_reserves"], return_geometry=False)
    reserve_df = pd.DataFrame([item.get("attributes", {}) for item in reserve_features])
    print({
        "fields": len(field_gdf), "discoveries": len(discovery_gdf),
        "facilities": len(facility_gdf), "pipelines": len(pipeline_gdf),
        "reserve_records": len(reserve_df)
    })
except Exception as error:
    print("Public SODIR data unavailable:", error)
    field_gdf = discovery_gdf = facility_gdf = pipeline_gdf = gpd.GeoDataFrame(
        {"geometry": []}, geometry="geometry", crs="EPSG:4326"
    )
    reserve_df = pd.DataFrame()

## Schema-resilient column selection

Public schemas can evolve. The helper searches normalized column names rather than assuming that every identifier remains unchanged.

In [ ]:
def find_column(frame, *candidate_fragments):
    normalized = {str(col).lower().replace("_", ""): col for col in frame.columns}
    for fragment in candidate_fragments:
        key = fragment.lower().replace("_", "")
        exact = [original for norm, original in normalized.items() if norm == key]
        if exact:
            return exact[0]
    for fragment in candidate_fragments:
        key = fragment.lower().replace("_", "")
        partial = [original for norm, original in normalized.items() if key in norm]
        if partial:
            return partial[0]
    return None

schema_columns = {
    "field_name": find_column(field_gdf, "fldName", "fieldName", "name"),
    "discovery_name": find_column(discovery_gdf, "dscName", "discoveryName", "name"),
    "facility_name": find_column(facility_gdf, "fclName", "facilityName", "name"),
    "pipeline_name": find_column(pipeline_gdf, "pipName", "pipelineName", "name"),
}
display(pd.Series(schema_columns, name="selected_column").to_frame())

## Public resource context

SODIR's Resource Accounts 2025 report that production remains high, reserve growth replaced a substantial share of production, and contingent resources in discoveries increased. This motivates area-development studies but does not justify any individual project.

The fictitious discovery is deliberately not calibrated to a named SODIR discovery.

In [ ]:
public_context = pd.DataFrame([
    ["NCS total liquids incl. sold/delivered", 9037, "million Sm3", "SODIR Resource Accounts 2025"],
    ["NCS total gas incl. sold/delivered", 6685, "billion Sm3", "SODIR Resource Accounts 2025"],
    ["Remaining oil reserves", 815, "million Sm3", "SODIR discovered resources 2025"],
    ["Remaining gas reserves", 1219, "billion Sm3", "SODIR discovered resources 2025"],
], columns=["metric", "value", "unit", "source"])
display(public_context)

## Gassco public pipeline snapshot

The following dimensions are transcribed from Gassco's public transport map and retained with source labels. They describe the large transport context—not the fictitious subsea tieback.

In [ ]:
gassco_pipelines = pd.DataFrame([
    ["OGT", "Oseberg", "Heimdal", 110, 36, "public Gassco transport map"],
    ["Kvitebjørn Pipeline", "Kvitebjørn", "Kollsnes", 148, 30, "public Gassco transport map"],
    ["Gjøa gas pipe", "Gjøa", "FLAGS", 131, 28, "public Gassco transport map"],
    ["Utsira High Gas Pipeline", "Edvard Grieg", "SAGE", 94, 16, "public Gassco transport map"],
    ["Polarled", "Aasta Hansteen", "Nyhamna", 482, 36, "public Gassco transport map"],
    ["Åsgard Transport", "Åsgard ESB", "Kårstø", 720, 42, "public Gassco transport map"],
    ["Tampen Link", "Statfjord", "FLAGS", 24, 32, "public Gassco transport map"],
], columns=["system", "from", "to", "length_km", "diameter_in", "source"])
display(gassco_pipelines)

## Gassco tariff and booking context

Gassco currently links a 2026 tariff view and long-range tariff forecast. The public tariff page also shows a SEGAL exit example of GBP 0.01555/Sm3 for gas year 2025. That value is not applied to Aurora Sør because it is route- and period-specific.

The economic model uses a clearly synthetic gas tariff. A real route must combine applicable entry, processing, exit, quality-service and capacity-fee terms with booked quantities and escalation.

In [ ]:
gassco_context = pd.DataFrame([
    ["Primary capacity", "Within Day / Short / Medium / Long Term", "booking constraint"],
    ["Secondary capacity", "buy and sell capacity", "alternative availability"],
    ["Interruptible concept", "defined in booking terms", "availability/risk scenario"],
    ["Nominations versus bookings", "comparison available to authorized shippers", "operational gate"],
    ["Tariff areas", "A-I Gassled plus other areas", "route-specific cost structure"],
    ["SEGAL public example", 0.01555, "GBP/Sm3, GY2025; not used in case"],
], columns=["item", "public_description_or_value", "study_use"])
display(gassco_context)

## 3. Fictitious discovery and map basis

Aurora Sør is placed in the northern North Sea solely to demonstrate an area workflow. The coordinate, water depth and route lengths are synthetic. Nearby public fields and facilities are shown only as geographic context.

In [ ]:
AURORA = {
    "name": "Aurora Sør (fictitious)",
    "longitude": 2.55,
    "latitude": 60.78,
    "water_depth_m": 300.0,
    "reservoir_top_m_tvdss": 3100.0,
    "reservoir_temperature_c": 90.0,
    "initial_pressure_bara": 330.0,
}
aurora_point = gpd.GeoDataFrame(
    [AURORA], geometry=[Point(AURORA["longitude"], AURORA["latitude"])], crs="EPSG:4326"
)
display(pd.Series(AURORA, name="value").to_frame())

## Geodesic distance helper

Geodesic distances are appropriate for initial ranking. Engineering route length will be longer because of bathymetry, seabed hazards, existing infrastructure, crossings, approach corridors and installation constraints.

In [ ]:
GEOD = Geod(ellps="WGS84")

def geodesic_km(lon1, lat1, lon2, lat2):
    _, _, distance_m = GEOD.inv(float(lon1), float(lat1), float(lon2), float(lat2))
    return distance_m / 1000.0

def projected_distance_km(left_gdf, right_gdf, crs=UTM_NORTH_SEA):
    left = left_gdf.to_crs(crs)
    right = right_gdf.to_crs(crs)
    return left.geometry.iloc[0].distance(right.geometry.iloc[0]) / 1000.0

print("Distance function check [km]:", geodesic_km(2.55, 60.78, 2.80, 60.90))

## Field-centroid screening

Distances to field polygons are measured to projected centroids. A true tie-in distance must be measured to the selected connection point, not to a field centroid.

In [ ]:
nearest_fields = pd.DataFrame()
field_name_col = schema_columns["field_name"]
if not field_gdf.empty and field_name_col:
    valid_fields = field_gdf.dropna(subset=["geometry"]).copy()
    projected = valid_fields.to_crs(UTM_NORTH_SEA)
    centroids = projected.geometry.centroid
    aurora_xy = aurora_point.to_crs(UTM_NORTH_SEA).geometry.iloc[0]
    valid_fields["screening_distance_km"] = centroids.distance(aurora_xy) / 1000.0
    nearest_fields = valid_fields[[field_name_col, "screening_distance_km"]].sort_values(
        "screening_distance_km"
    ).head(15)
display(nearest_fields)

## Facility-distance screening

Facilities are more relevant than field centroids for a host long list, but public facility presence says nothing about processing compatibility or spare capacity.

In [ ]:
nearest_facilities = pd.DataFrame()
facility_name_col = schema_columns["facility_name"]
if not facility_gdf.empty and facility_name_col:
    valid_facilities = facility_gdf.dropna(subset=["geometry"]).copy()
    valid_facilities = valid_facilities[valid_facilities.geometry.geom_type == "Point"]
    aurora_xy = aurora_point.to_crs(UTM_NORTH_SEA).geometry.iloc[0]
    valid_xy = valid_facilities.to_crs(UTM_NORTH_SEA)
    valid_facilities["screening_distance_km"] = valid_xy.geometry.distance(aurora_xy) / 1000.0
    nearest_facilities = valid_facilities[[facility_name_col, "screening_distance_km"]].sort_values(
        "screening_distance_km"
    ).head(20)
display(nearest_facilities)

## Static NCS area map

The first map is intentionally simple and printable. It shows public field/pipeline/facility geometry around the fictitious coordinate.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
bounds = (0.0, 58.5, 5.5, 63.0)

if not field_gdf.empty:
    field_gdf.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]].plot(
        ax=ax, facecolor="none", edgecolor="steelblue", linewidth=0.6, alpha=0.7
    )
if not pipeline_gdf.empty:
    pipeline_gdf.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]].plot(
        ax=ax, color="darkorange", linewidth=0.8, alpha=0.7
    )
if not facility_gdf.empty:
    facility_gdf.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]].plot(
        ax=ax, color="black", markersize=8, alpha=0.7
    )
aurora_point.plot(ax=ax, color="crimson", marker="*", markersize=180, label=AURORA["name"])
ax.set(xlim=(bounds[0], bounds[2]), ylim=(bounds[1], bounds[3]),
       xlabel="Longitude", ylabel="Latitude", title="Public NCS context and fictitious Aurora Sør")
ax.legend()
plt.show()

## Interactive Folium map

Folium allows inspection of public features and later route overlays. Geometry is simplified only for browser performance.

In [ ]:
area_map = folium.Map(
    location=[AURORA["latitude"], AURORA["longitude"]],
    zoom_start=7,
    tiles="CartoDB positron"
)
folium.Marker(
    [AURORA["latitude"], AURORA["longitude"]],
    tooltip=AURORA["name"],
    icon=folium.Icon(color="red", icon="star")
).add_to(area_map)

if not field_gdf.empty:
    subset = field_gdf.cx[0.0:5.5, 58.5:63.0].copy()
    if not subset.empty:
        folium.GeoJson(
            json.loads(subset.to_json()),
            name="SODIR fields",
            style_function=lambda _: {"color": "#3274a1", "weight": 1, "fillOpacity": 0.05}
        ).add_to(area_map)
if not pipeline_gdf.empty:
    subset = pipeline_gdf.cx[0.0:5.5, 58.5:63.0].copy()
    if not subset.empty:
        folium.GeoJson(
            json.loads(subset.to_json()),
            name="SODIR pipelines",
            style_function=lambda _: {"color": "#e1812c", "weight": 2}
        ).add_to(area_map)
folium.LayerControl().add_to(area_map)
area_map

## Candidate host abstraction

The lifecycle model uses synthetic Host A and Host B so public asset names are never assigned invented capacities. Their nominal distances are engineering assumptions, even if public geometry suggests nearby infrastructure.

In [ ]:
host_candidates = gpd.GeoDataFrame([
    {"host": "Synthetic Host A", "longitude": 2.86, "latitude": 60.96,
     "route_basis": "direct brownfield candidate", "assumed_tieback_km": 35.0},
    {"host": "Synthetic Host B", "longitude": 3.34, "latitude": 60.58,
     "route_basis": "remote brownfield candidate", "assumed_tieback_km": 65.0},
    {"host": "Standalone FPSO", "longitude": 2.63, "latitude": 60.82,
     "route_basis": "new facility", "assumed_tieback_km": 12.0},
], geometry=[Point(2.86, 60.96), Point(3.34, 60.58), Point(2.63, 60.82)], crs="EPSG:4326")

host_candidates["geodesic_km"] = host_candidates.apply(
    lambda row: geodesic_km(AURORA["longitude"], AURORA["latitude"], row.longitude, row.latitude), axis=1
)
host_candidates["route_factor"] = host_candidates.assumed_tieback_km / host_candidates.geodesic_km
display(host_candidates.drop(columns="geometry"))

## Route-length reconciliation

Route factors above 1.0 represent approach corridors, topology and installation allowances. Large discrepancies are a prompt to revisit both coordinates and engineering assumptions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
route_compare = host_candidates.set_index("host")[["geodesic_km", "assumed_tieback_km"]]
route_compare.plot.bar(ax=ax)
ax.set(ylabel="Distance [km]", title="Geodesic versus assumed engineering route")
ax.tick_params(axis="x", rotation=20)
plt.show()

## Synthetic seabed exclusion zone

The exclusion polygon below is purely educational. It demonstrates why direct distance is not necessarily route length. Real routing needs bathymetry, slopes, soil, boulders, wrecks, protected areas, fisheries, UXO, crossings and installation limits.

In [ ]:
aurora_utm = aurora_point.to_crs(UTM_NORTH_SEA)
host_utm = host_candidates.to_crs(UTM_NORTH_SEA)
a_xy = aurora_utm.geometry.iloc[0]
h_xy = host_utm.loc[host_candidates.host == "Synthetic Host A", "geometry"].iloc[0]

midpoint = a_xy.interpolate(0.50, normalized=True) if hasattr(a_xy, "interpolate") else Point(
    (a_xy.x + h_xy.x) / 2, (a_xy.y + h_xy.y) / 2
)
# Point has no path interpolation; use the line explicitly.
direct_a = LineString([a_xy, h_xy])
midpoint = direct_a.interpolate(0.52, normalized=True)
exclusion_zone = midpoint.buffer(3500.0)
print("Synthetic exclusion-zone area [km2]:", exclusion_zone.area / 1e6)

## Screening route around the exclusion

A waypoint outside the exclusion creates a simple routed alternative. This is not an optimizer; it demonstrates the topology needed for later route engineering.

In [ ]:
offset = 5500.0
waypoint = Point(midpoint.x - offset, midpoint.y + offset)
routed_a = LineString([a_xy, waypoint, h_xy])

route_options = pd.DataFrame([
    ["Host A direct", direct_a.length / 1000.0, direct_a.intersects(exclusion_zone), "geometric screening"],
    ["Host A routed", routed_a.length / 1000.0, routed_a.intersects(exclusion_zone), "one-waypoint screening"],
    ["Host A lifecycle basis", 35.0, False, "synthetic NeqSim assumption"],
], columns=["route", "length_km", "crosses_exclusion", "basis"])
display(route_options)

## Plot routed subsea alternatives

Both route geometry and constraint geometry are plotted in UTM metres to preserve local distances.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
gpd.GeoSeries([exclusion_zone], crs=UTM_NORTH_SEA).plot(ax=ax, color="mistyrose", edgecolor="red", alpha=0.6)
gpd.GeoSeries([direct_a], crs=UTM_NORTH_SEA).plot(ax=ax, color="grey", linestyle="--", label="direct")
gpd.GeoSeries([routed_a], crs=UTM_NORTH_SEA).plot(ax=ax, color="green", linewidth=2, label="routed")
gpd.GeoSeries([a_xy, h_xy, waypoint], crs=UTM_NORTH_SEA).plot(ax=ax, color=["crimson", "black", "green"])
ax.set(title="Educational subsea route around a synthetic exclusion zone", xlabel="UTM easting [m]", ylabel="UTM northing [m]")
ax.legend()
plt.show()

## Pipeline endpoint topology from public data

When public pipeline attributes expose from/to facility names, they can seed a graph. If attributes are absent, geometry endpoints can still be used for spatial topology, but asset identity requires reconciliation.

In [ ]:
pipeline_name_col = schema_columns["pipeline_name"]
from_col = find_column(pipeline_gdf, "pipFromFacilityName", "fromFacility", "from")
to_col = find_column(pipeline_gdf, "pipToFacilityName", "toFacility", "to")
length_col = find_column(pipeline_gdf, "pipLength", "length")

public_graph = nx.Graph()
if from_col and to_col and not pipeline_gdf.empty:
    for _, row in pipeline_gdf.iterrows():
        origin, destination = row.get(from_col), row.get(to_col)
        if pd.notna(origin) and pd.notna(destination):
            length = row.get(length_col, np.nan) if length_col else np.nan
            public_graph.add_edge(str(origin), str(destination), length_km=float(length) if pd.notna(length) else np.nan,
                                  source="SODIR")
print("Public topology nodes/edges:", public_graph.number_of_nodes(), public_graph.number_of_edges())

## Conceptual gas-export topology

The graph below joins synthetic development choices to named public Gassco corridors. The connection edges are scenarios, not statements that tie-in rights or capacity exist.

In [ ]:
gas_graph = nx.DiGraph()
gas_edges = [
    ("Aurora Sør", "Synthetic Host A", 35.0, "synthetic tieback"),
    ("Synthetic Host A", "OGT reference corridor", 25.0, "synthetic connection"),
    ("OGT reference corridor", "Heimdal", 110.0, "Gassco public OGT length"),
    ("Heimdal", "European receiving route", 360.0, "public transport context"),
    ("Aurora Sør", "Synthetic Host B", 65.0, "synthetic tieback"),
    ("Synthetic Host B", "Kollsnes reference route", 55.0, "synthetic connection"),
    ("Kollsnes reference route", "Kollsnes", 148.0, "Gassco Kvitebjørn reference length"),
    ("Kollsnes", "European receiving route", 300.0, "public Zeepipe context"),
    ("Aurora Sør", "Standalone FPSO", 12.0, "synthetic infield route"),
    ("Standalone FPSO", "Dedicated gas lateral", 90.0, "synthetic lateral"),
    ("Dedicated gas lateral", "European receiving route", 300.0, "screening continuation"),
]
for origin, destination, length, basis in gas_edges:
    gas_graph.add_edge(origin, destination, length_km=length, basis=basis)

for target in ["Synthetic Host A", "Synthetic Host B", "Standalone FPSO"]:
    path = nx.shortest_path(gas_graph, "Aurora Sør", target, weight="length_km")
    distance = nx.path_weight(gas_graph, path, weight="length_km")
    print(target, path, distance, "km")

## Visualize topology

A topology graph complements the map: it shows decision nodes, shared systems and downstream dependencies. It should eventually be linked to equipment identifiers and commercial areas.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
pos = nx.spring_layout(gas_graph, seed=RANDOM_SEED, k=1.2)
nx.draw_networkx(gas_graph, pos=pos, ax=ax, node_size=1300, node_color="aliceblue",
                 edge_color="slategrey", font_size=8, arrows=True)
edge_labels = {(u, v): f"{d['length_km']:.0f} km" for u, v, d in gas_graph.edges(data=True)}
nx.draw_networkx_edge_labels(gas_graph, pos=pos, edge_labels=edge_labels, font_size=7, ax=ax)
ax.set_title("Conceptual gas-export dependency graph")
ax.axis("off")
plt.show()

## Map-screening conclusion

Host A is retained as the short brownfield case, Host B as the remote alternative, and a new FPSO as the standalone case. The selection is synthetic, while the workflow for finding and mapping public candidates is reusable.

Before concept select, replace assumed routes with surveyed corridors and reconcile every candidate to an operator-approved connection point.

## 4. Decision basis and synthetic assumptions

The decision basis is frozen before running the mutable models. This prevents one concept from receiving a hidden advantage.

In [ ]:
decision_basis = pd.DataFrame([
    ["Reservoir temperature", 90.0, "degC", "synthetic"],
    ["Initial reservoir pressure", 330.0, "bara", "synthetic"],
    ["Executable oil inventory", 45.0, "million Sm3", "NeqSim reference model"],
    ["Executable gas inventory", 180.0, "million Sm3", "NeqSim reference model"],
    ["Executable water inventory", 90.0, "million Sm3", "NeqSim reference model"],
    ["Water depth", 300.0, "m", "synthetic"],
    ["Producers", 6, "count", "synthetic"],
    ["Gas injectors", 3, "count", "synthetic"],
    ["First oil", 2029, "calendar year", "synthetic"],
    ["Availability", 0.92, "fraction", "synthetic"],
    ["Oil price", 75.0, "USD/bbl", "synthetic"],
    ["Gas price", 0.28, "USD/Sm3", "synthetic"],
], columns=["parameter", "value", "unit", "classification"])
display(decision_basis)

## Resource uncertainty versus executable inventory

The screening FieldConcept carries a 550/700/850 MMbbl low/base/high resource range, while the executable SimpleReservoir is initialized with explicit fluid inventories. They serve different purposes and must not be silently treated as identical.

In a project model, reconcile volumetrics, PVT reference conditions and recovery definition before economics. Here the executable lifecycle—not the screening range—determines production.

In [ ]:
resource_range = pd.DataFrame({
    "case": ["low", "base", "high"],
    "screening_resource_mmbbl": [550.0, 700.0, 850.0],
})
resource_range["screening_resource_msm3"] = resource_range.screening_resource_mmbbl / BARREL_PER_SM3
resource_range["illustrative_recoverable_msm3_at_48pct"] = resource_range.screening_resource_msm3 * 0.48
display(resource_range)

## Concept definitions

Standalone injection uses an FPSO with detailed oil, gas, water and injection processing. Natural depletion removes injection. Brownfield routes use producing host profiles, allocation policy, holdback and different tieback distances.

In [ ]:
concept_catalogue = pd.DataFrame([
    ["Standalone FPSO", "greenfield", 12.0, "gas injection", "new detailed process"],
    ["Natural depletion", "greenfield", 12.0, "gas export", "new detailed process"],
    ["Host A priority", "brownfield", 35.0, "no injection", "host production first"],
    ["Host A managed", "brownfield", 35.0, "no injection", "pro-rata + planned holdback"],
    ["Host B remote", "brownfield", 65.0, "no injection", "mature remote host"],
], columns=["concept", "development_mode", "tieback_km", "reservoir_support", "capacity_policy"])
display(concept_catalogue)

## Project stage and estimate class

Well and SURF costs use NeqSim engineering estimators. Topsides and project costs in the reference factory are parametric Class-4 allowances. This is appropriate for learning and early option ranking, not sanction.

In [ ]:
maturity_gates = pd.DataFrame([
    ["DG0 opportunity", "broad resource/routing range", "screening only"],
    ["DG1 feasibility", "PVT, wells, routes, capacities, costs", "this notebook level"],
    ["DG2 concept select", "operator-validated models and agreements", "additional work"],
    ["Sanction/PDO", "defined design, assurance, execution and commercial basis", "outside scope"],
], columns=["stage", "minimum_evidence", "status"])
display(maturity_gates)

## Common economic basis

All options use the same base prices and discount rate. Route-specific CAPEX, OPEX and tariffs remain different because they represent different development architectures.

In [ ]:
economic_basis = pd.Series({
    "oil_price_usd_bbl": 75.0,
    "gas_price_usd_sm3": 0.28,
    "real_discount_rate": 0.08,
    "grid_factor_kg_co2_kwh": 0.018,
    "currency_basis": "real MUSD (synthetic)",
    "tax_basis": "NeqSim Norwegian tax model",
})
display(economic_basis.to_frame("value"))

## Quality basis

The factory activates CO2, H2S, oxygen, energy-content, oil-RVP, BS&W and oil-in-water limits. These are teaching placeholders. Gas limits must be replaced by the applicable delivery-point agreement.

In [ ]:
quality_basis = pd.DataFrame([
    ["Gas CO2", 2.5, "mol% max"],
    ["Gas H2S", 5.0, "ppm max"],
    ["Gas oxygen", 0.0002, "mol% max"],
    ["Gas GCV", "38.1-43.7", "MJ/Sm3"],
    ["Gas Wobbe", "48.3-52.8", "MJ/Sm3"],
    ["Gas relative density", 0.70, "max"],
    ["Oil RVP", 1.0, "bara max"],
    ["Oil BS&W", 0.5, "vol% max"],
    ["Oil in water", 30.0, "mg/L max"],
], columns=["property", "synthetic_limit", "basis"])
display(quality_basis)

## Decision gates

An option must pass technical integrity, product/discharge compliance, host and export access, execution schedule and commercial eligibility before NPV ranking. NPV cannot rescue an infeasible route.

In [ ]:
decision_gates = pd.DataFrame([
    ["Subsurface", "deliverability and recoverable range", "open until appraisal"],
    ["Wells/SURF", "pressure, integrity, flow assurance, installability", "screened"],
    ["Facility", "detailed capacity and modifications", "screened"],
    ["Products", "oil/gas/water specifications", "calculated placeholders"],
    ["Host", "capacity, consent, shutdown and metering", "synthetic only"],
    ["Gas transport", "booked route/capacity/tariff", "not secured"],
    ["Economics", "NPV, IRR, break-even and robustness", "calculated"],
], columns=["gate", "evidence", "notebook_status"])
display(decision_gates)

## 5. Compositional PVT model

The same Peng–Robinson compositional fluid feeds the reservoir and process. Heavy fractions are characterized as TBP fractions, water is included and multiphase checking is enabled.

In [ ]:
base_fluid = NorwegianOilFieldCase.createReservoirFluid()
base_fluid.setTemperature(AURORA["reservoir_temperature_c"], "C")
base_fluid.setPressure(AURORA["initial_pressure_bara"], "bara")
ThermodynamicOperations(base_fluid).TPflash()
base_fluid.initPhysicalProperties()
print("Phases at reservoir conditions:", base_fluid.getNumberOfPhases())

## Fluid composition

Component amounts below are model inputs, not a normalized laboratory analysis. NeqSim normalizes the phase/system composition internally.

In [ ]:
components = []
phase0 = base_fluid.getPhase(0)
for index in range(phase0.getNumberOfComponents()):
    component = phase0.getComponent(index)
    components.append({
        "component": str(component.getName()),
        "overall_z": float(component.getz()),
        "phase0_x": float(component.getx()),
    })
composition_df = pd.DataFrame(components)
display(composition_df)
print("Sum overall z:", composition_df.overall_z.sum())

## Composition visualization

A logarithmic axis makes light and heavy contributions visible together. Pseudocomponents are essential for oil density, phase behavior and stabilization.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_comp = composition_df[composition_df.overall_z > 0].copy()
ax.bar(plot_comp.component, plot_comp.overall_z)
ax.set_yscale("log")
ax.set(ylabel="Overall mole fraction [-]", title="Synthetic Aurora Sør fluid")
ax.tick_params(axis="x", rotation=70)
plt.show()

## Reservoir-condition phase properties

Density and viscosity are calculated from the flashed NeqSim phases. These properties connect PVT to well and pipeline pressure drop.

In [ ]:
phase_rows = []
for index in range(base_fluid.getNumberOfPhases()):
    phase = base_fluid.getPhase(index)
    phase_rows.append({
        "index": index,
        "type": str(phase.getType()),
        "beta": float(base_fluid.getBeta(index)),
        "density_kg_m3": float(phase.getDensity("kg/m3")),
        "viscosity_kg_m_s": float(phase.getViscosity("kg/msec")),
        "molar_mass_kg_mol": float(phase.getMolarMass()),
    })
phase_properties = pd.DataFrame(phase_rows)
display(phase_properties)

## Pressure-depletion PVT sweep

A TP flash is performed at every pressure. Gas-phase fraction indicates liberation during depletion. This is diagnostic; a PVT model should be regressed to laboratory tests.

In [ ]:
pvt_rows = []
for pressure_bara in np.linspace(360.0, 40.0, 33):
    fluid = NorwegianOilFieldCase.createReservoirFluid()
    fluid.setTemperature(90.0, "C")
    fluid.setPressure(float(pressure_bara), "bara")
    ThermodynamicOperations(fluid).TPflash()
    fluid.initPhysicalProperties()
    gas_beta = sum(
        float(fluid.getBeta(i))
        for i in range(fluid.getNumberOfPhases())
        if "gas" in str(fluid.getPhase(i).getType()).lower()
    )
    oil_density = np.nan
    for i in range(fluid.getNumberOfPhases()):
        if "oil" in str(fluid.getPhase(i).getType()).lower():
            oil_density = float(fluid.getPhase(i).getDensity("kg/m3"))
    pvt_rows.append({
        "pressure_bara": pressure_bara,
        "phase_count": int(fluid.getNumberOfPhases()),
        "gas_beta": gas_beta,
        "oil_density_kg_m3": oil_density,
    })
pvt_sweep = pd.DataFrame(pvt_rows)
display(pvt_sweep.head())

## Pressure-sweep discussion

The phase transition controls producing GOR, wellstream hydraulics, gas-compression load and injection availability. A fluid error therefore propagates into both technical and economic results.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].step(pvt_sweep.pressure_bara, pvt_sweep.phase_count, where="mid")
axes[1].plot(pvt_sweep.pressure_bara, pvt_sweep.gas_beta)
axes[2].plot(pvt_sweep.pressure_bara, pvt_sweep.oil_density_kg_m3)
for ax in axes:
    ax.invert_xaxis()
    ax.set_xlabel("Pressure [bara]")
axes[0].set_ylabel("Phase count")
axes[1].set_ylabel("Gas beta [-]")
axes[2].set_ylabel("Oil density [kg/m3]")
plt.tight_layout()
plt.show()

## Temperature sensitivity

A temperature sweep at a representative pressure demonstrates the coupling between thermal assumptions and phase split.

In [ ]:
temperature_rows = []
for temperature_c in np.linspace(20.0, 120.0, 21):
    fluid = NorwegianOilFieldCase.createReservoirFluid()
    fluid.setTemperature(float(temperature_c), "C")
    fluid.setPressure(150.0, "bara")
    ThermodynamicOperations(fluid).TPflash()
    gas_beta = sum(
        float(fluid.getBeta(i))
        for i in range(fluid.getNumberOfPhases())
        if "gas" in str(fluid.getPhase(i).getType()).lower()
    )
    temperature_rows.append({"temperature_c": temperature_c, "gas_beta": gas_beta,
                             "phase_count": int(fluid.getNumberOfPhases())})
temperature_sweep = pd.DataFrame(temperature_rows)
display(temperature_sweep)

## Surface separation diagnostic

The detailed process performs its own flashes. This single separator diagnostic helps explain why pressure staging recovers oil and changes compression duty.

In [ ]:
separator_rows = []
for pressure_bara in [55.0, 20.0, 5.0, 1.5]:
    fluid = NorwegianOilFieldCase.createReservoirFluid()
    fluid.setTemperature(50.0, "C")
    fluid.setPressure(pressure_bara, "bara")
    ThermodynamicOperations(fluid).TPflash()
    betas = {str(fluid.getPhase(i).getType()): float(fluid.getBeta(i))
             for i in range(fluid.getNumberOfPhases())}
    separator_rows.append({"pressure_bara": pressure_bara, "phases": fluid.getNumberOfPhases(), **betas})
separator_diagnostic = pd.DataFrame(separator_rows).fillna(0.0)
display(separator_diagnostic)

## PVT assurance checklist

A project PVT package should document sample representativity, recombination, EOS selection, heavy-end characterization, CCE/DL/CVD/separator regression, viscosity and density errors, saturation pressure, water/brine chemistry and uncertainty.

The educational model demonstrates integration, not laboratory matching.

In [ ]:
pvt_assurance = pd.DataFrame([
    ["Composition QA", "synthetic only", "replace with recombined sample"],
    ["Saturation pressure", "diagnostic sweep", "regress to experiment"],
    ["Density/viscosity", "EOS correlations", "validate over P/T range"],
    ["Separator test", "illustrative flashes", "match measured stages"],
    ["Brine chemistry", "not characterized", "required for scale/corrosion"],
], columns=["item", "current_evidence", "next_action"])
display(pvt_assurance)

## 6. Reservoir configuration

The reference lifecycle uses a tank-style material-balance reservoir with explicit oil, gas and water inventories and pressure depletion. It is suitable for integrated concept comparison, not a replacement for a 3D simulation.

In [ ]:
greenfield_concept = NorwegianOilFieldCase.createGasInjectionCase()
configuration = greenfield_concept.getConfiguration()
model = greenfield_concept.getModel()
reservoir = model.getReservoir()

configuration_summary = pd.Series({
    "start_year": configuration.getStartYear(),
    "project_years": configuration.getProjectYears(),
    "time_step_days": configuration.getTimeStepDays(),
    "availability": configuration.getAvailability(),
    "producer_count": configuration.getProducerCount(),
    "PI_per_well_sm3_d_bar": configuration.getProductivityIndexSm3PerDayBarPerWell(),
    "minimum_bhp_bara": configuration.getMinimumBottomHolePressureBara(),
    "plateau_oil_sm3_d": configuration.getPlateauOilRateSm3PerDay(),
    "economic_limit_sm3_d": configuration.getEconomicLimitOilRateSm3PerDay(),
})
display(configuration_summary.to_frame("value"))

## Water-cut model

Water cut remains at the initial value until breakthrough and then ramps linearly. This simple assumption is transparent but can dominate late-life water handling and host availability.

In [ ]:
ages = np.linspace(0.0, configuration.getProjectYears(), 101)
watercut_df = pd.DataFrame({
    "field_age_years": ages,
    "water_cut": [configuration.getWaterCut(float(age)) for age in ages],
})
display(watercut_df.iloc[::10])

## Water-cut visualization

Water-processing constraints typically arrive after oil plateau. Brownfield route ranking can reverse if host water capacity is tight.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(watercut_df.field_age_years, 100.0 * watercut_df.water_cut)
ax.set(xlabel="Field age [years]", ylabel="Water cut [%]", title="Synthetic water-cut schedule")
plt.show()

## Well deliverability envelope

The screening deliverability is producer count times PI times reservoir-pressure drawdown, capped by the plateau. Detailed well/network models can replace this provider.

In [ ]:
pressure_grid = np.linspace(330.0, 180.0, 31)
deliverability = pd.DataFrame({"reservoir_pressure_bara": pressure_grid})
deliverability["pi_limit_sm3_d"] = (
    configuration.getProducerCount()
    * configuration.getProductivityIndexSm3PerDayBarPerWell()
    * np.maximum(pressure_grid - configuration.getMinimumBottomHolePressureBara(), 0.0)
)
deliverability["plateau_limited_sm3_d"] = np.minimum(
    deliverability.pi_limit_sm3_d, configuration.getPlateauOilRateSm3PerDay()
)
display(deliverability.head())

## Deliverability visualization

The curve makes the economic interaction clear: reservoir support can extend plateau, but only if wells, SURF and facility can accept the supported rate.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(deliverability.reservoir_pressure_bara, deliverability.pi_limit_sm3_d, label="PI limit")
ax.plot(deliverability.reservoir_pressure_bara, deliverability.plateau_limited_sm3_d, label="after plateau cap")
ax.invert_xaxis()
ax.set(xlabel="Reservoir pressure [bara]", ylabel="Oil potential [Sm3/d]")
ax.legend()
plt.show()

## Gas-injection strategy

The reference case recycles up to 85% of recovered gas, limited by injection compression and well injectivity. Export gas is the residual after allocation.

In [ ]:
injection_basis = pd.Series({
    "start_age_years": configuration.getGasInjectionStartYear(),
    "recycle_fraction": configuration.getProducedGasRecycleFraction(),
    "surface_capacity_sm3_d": configuration.getMaximumGasInjectionRateSm3PerDay(),
    "injector_count": 3,
    "injection_pressure_bara": 350.0,
    "fracture_pressure_bara": 410.0,
})
display(injection_basis.to_frame("value"))

## Detailed gas-injector calculation

NeqSim's InjectionWellModel estimates achievable injection rate from formation and tubing assumptions while respecting maximum BHP and fracture pressure.

In [ ]:
injector = (
    InjectionWellModel(InjectionType.GAS_INJECTOR)
    .setReservoirPressure(330.0, "bara")
    .setReservoirTemperature(90.0, "C")
    .setFormationPermeability(180.0, "mD")
    .setFormationThickness(45.0, "m")
    .setSkinFactor(2.0)
    .setWellDepth(3100.0, "m")
    .setTubingID(0.127, "m")
    .setMaxBHP(390.0, "bara")
    .setFracturePressure(410.0, "bara")
    .setSurfaceInjectionPressure(350.0, "bara")
)
injectivity_result = injector.calculateMaximumRate()
injectivity = pd.Series({
    "per_well_achievable_rate": injectivity_result.getAchievableRate(),
    "three_well_capacity": 3.0 * injectivity_result.getAchievableRate(),
    "configured_surface_capacity": configuration.getMaximumGasInjectionRateSm3PerDay(),
})
display(injectivity.to_frame("Sm3/d"))

## Injectivity uncertainty

Permeability, skin, reservoir pressure and completion condition change through life. The screening below varies permeability while retaining other assumptions.

In [ ]:
injectivity_rows = []
for permeability_md in [50, 100, 180, 300, 500]:
    test_well = (
        InjectionWellModel(InjectionType.GAS_INJECTOR)
        .setReservoirPressure(330.0, "bara").setReservoirTemperature(90.0, "C")
        .setFormationPermeability(float(permeability_md), "mD").setFormationThickness(45.0, "m")
        .setSkinFactor(2.0).setWellDepth(3100.0, "m").setTubingID(0.127, "m")
        .setMaxBHP(390.0, "bara").setFracturePressure(410.0, "bara")
        .setSurfaceInjectionPressure(350.0, "bara")
    )
    result = test_well.calculateMaximumRate()
    injectivity_rows.append({"permeability_md": permeability_md,
                             "rate_per_well_sm3_d": result.getAchievableRate()})
injectivity_sensitivity = pd.DataFrame(injectivity_rows)
display(injectivity_sensitivity)

## Reservoir uncertainty sample

This probabilistic sample is a transparent teaching layer around the screening resource range. It is not coupled to NeqSim's mutable reservoir and therefore must not be presented as a probabilistic production forecast.

In [ ]:
resource_samples_mmbbl = rng.triangular(550.0, 700.0, 850.0, 5000)
recovery_samples = np.clip(rng.normal(0.48, 0.06, 5000), 0.25, 0.65)
recoverable_samples_msm3 = resource_samples_mmbbl / BARREL_PER_SM3 * recovery_samples
uncertainty_summary = pd.Series({
    "P10_recoverable_msm3": np.quantile(recoverable_samples_msm3, 0.10),
    "P50_recoverable_msm3": np.quantile(recoverable_samples_msm3, 0.50),
    "P90_recoverable_msm3": np.quantile(recoverable_samples_msm3, 0.90),
})
display(uncertainty_summary.to_frame("value"))

## Reservoir-model upgrade path

A real study should connect Eclipse/OPM or another reservoir schedule through FieldProductionPotentialProvider, or build an appropriately validated material-balance model. Preserve pressure, phase rates, injection, voidage and uncertainty provenance for every time step.

In [ ]:
reservoir_upgrade = pd.DataFrame([
    ["Screening tank", "current", "integrated concept comparison"],
    ["Material balance history match", "next", "appraisal/dynamic data"],
    ["3D reservoir schedule", "next", "well-by-well constraints"],
    ["Closed-loop coupled model", "advanced", "process backpressure and injection feedback"],
], columns=["model_level", "study_position", "purpose"])
display(reservoir_upgrade)

## 7. Wells and SURF process topology

The reference ProcessSystem includes reservoir streams, aggregate production tubing, multiphase flowline, separation, water treatment, oil pumping, gas recovery, export compression and injection compression.

In [ ]:
process_system = model.getProcessSystem()
unit_rows = []
for index, unit in enumerate(process_system.getUnitOperations()):
    unit_rows.append({
        "sequence": index,
        "name": str(unit.getName()),
        "java_class": str(unit.getClass().getSimpleName()),
    })
unit_table = pd.DataFrame(unit_rows)
display(unit_table)

## Equipment-class inventory

Counting equipment by class highlights the engineering scope and gaps. A professional greenfield model would normally add dehydration, gas dew-point control, metering, utilities, flare/relief interfaces and controls.

In [ ]:
equipment_inventory = (
    unit_table.groupby("java_class").size().rename("count").sort_values(ascending=False).to_frame()
)
display(equipment_inventory)

## Process dependency graph

The graph below is reconstructed from the named main path for explanation. NeqSim itself executes the connected stream topology.

In [ ]:
main_path = [
    "reservoir", "production wells", "aggregate tubing", "production flowline",
    "inlet choke", "HP separator", "LP separation", "oil export",
]
gas_path = ["HP/LP gas recovery", "gas allocation", "gas export compression", "export"]
injection_path = ["gas allocation", "injection stage 1", "intercooler", "injection stage 2", "reservoir"]

process_graph = nx.DiGraph()
nx.add_path(process_graph, main_path)
nx.add_path(process_graph, gas_path)
nx.add_path(process_graph, injection_path)
fig, ax = plt.subplots(figsize=(14, 6))
pos = nx.spring_layout(process_graph, seed=RANDOM_SEED)
nx.draw_networkx(process_graph, pos=pos, ax=ax, node_color="honeydew", node_size=1400, font_size=8, arrows=True)
ax.axis("off")
ax.set_title("Reservoir-to-export teaching topology")
plt.show()

## Inspect production tubing and flowline

The executable model uses aggregate hydraulic elements. Length and diameter are taken directly from the NeqSim equipment objects.

In [ ]:
tubing = process_system.getUnit("Aggregate production tubing")
flowline = process_system.getUnit("Production flowline")
surf_geometry = pd.DataFrame([
    ["Aggregate tubing", tubing.getLength(), tubing.getDiameter()],
    ["Production flowline", flowline.getLength(), flowline.getDiameter()],
], columns=["element", "length_m", "inside_diameter_m"])
display(surf_geometry)

## Production-flowline diameter screening

A separate NeqSim pipe calculation compares multiphase pressure loss across candidate diameters. It is an isolated screening calculation; the lifecycle factory retains its own 14-inch-equivalent route.

In [ ]:
def simulate_multiphase_pipe(length_km, diameter_m, mass_rate_kg_s=250.0):
    fluid = NorwegianOilFieldCase.createReservoirFluid()
    fluid.setPressure(100.0, "bara")
    fluid.setTemperature(55.0, "C")
    ThermodynamicOperations(fluid).TPflash()
    inlet = Stream(f"screening inlet {diameter_m}", fluid)
    inlet.setFlowRate(float(mass_rate_kg_s), "kg/sec")
    inlet.run()
    pipe = PipeBeggsAndBrills(f"screening pipe {diameter_m}", inlet)
    pipe.setLength(float(length_km) * 1000.0)
    pipe.setElevation(300.0)
    pipe.setDiameter(float(diameter_m))
    pipe.setPipeWallRoughness(1.5e-5)
    pipe.setNumberOfIncrements(20)
    pipe.run()
    return float(pipe.getOutletStream().getPressure("bara"))

multiphase_rows = []
for length_km in [12.0, 35.0, 65.0]:
    for diameter_m in [0.254, 0.305, 0.356, 0.406]:
        try:
            outlet = simulate_multiphase_pipe(length_km, diameter_m)
            multiphase_rows.append([length_km, diameter_m, outlet, 100.0 - outlet, "ok"])
        except Exception as error:
            multiphase_rows.append([length_km, diameter_m, np.nan, np.nan, str(error)[:80]])
multiphase_screen = pd.DataFrame(multiphase_rows, columns=[
    "length_km", "diameter_m", "outlet_bara", "pressure_drop_bar", "status"
])
display(multiphase_screen)

## Multiphase hydraulic visualization

Long routes and smaller diameters consume the pressure margin needed by wells and host tie-in. Hydrate, wax, slugging, restart and thermal behavior remain separate assurance gates.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for length_km, group in multiphase_screen.groupby("length_km"):
    ax.plot(group.diameter_m * 1000.0, group.pressure_drop_bar, marker="o", label=f"{length_km:.0f} km")
ax.set(xlabel="Inside diameter [mm]", ylabel="Pressure drop [bar]", title="NeqSim multiphase screening")
ax.legend(title="Route")
plt.show()

## Gas-lateral hydraulic model

Gas export alternatives are assessed with the same compositional model. This is a local subsea lateral, not the large Gassco trunkline.

In [ ]:
def simulate_gas_lateral(length_km, diameter_m, rate_sm3_d=2.5e6, inlet_bara=180.0):
    fluid = NorwegianOilFieldCase.createReservoirFluid()
    fluid.setPressure(55.0, "bara")
    fluid.setTemperature(40.0, "C")
    ThermodynamicOperations(fluid).TPflash()
    gas = fluid.phaseToSystem("gas")
    gas.setPressure(float(inlet_bara), "bara")
    gas.setTemperature(35.0, "C")
    inlet = Stream(f"gas lateral feed {diameter_m}", gas)
    inlet.setFlowRate(float(rate_sm3_d), "Sm3/day")
    inlet.run()
    pipe = PipeBeggsAndBrills(f"gas lateral {length_km} km {diameter_m} m", inlet)
    pipe.setLength(float(length_km) * 1000.0)
    pipe.setElevation(0.0)
    pipe.setDiameter(float(diameter_m))
    pipe.setPipeWallRoughness(1.5e-5)
    pipe.setNumberOfIncrements(30)
    pipe.run()
    return float(pipe.getOutletStream().getPressure("bara"))

gas_lateral_rows = []
for length_km in [35.0, 65.0, 90.0]:
    for diameter_m in [0.203, 0.254, 0.305, 0.356, 0.406]:
        try:
            outlet = simulate_gas_lateral(length_km, diameter_m)
            gas_lateral_rows.append([length_km, diameter_m, outlet, 180.0 - outlet])
        except Exception:
            gas_lateral_rows.append([length_km, diameter_m, np.nan, np.nan])
gas_lateral_screen = pd.DataFrame(gas_lateral_rows, columns=[
    "length_km", "diameter_m", "outlet_bara", "pressure_drop_bar"
])
display(gas_lateral_screen)

## Gas-pipeline solution comparison

Diameter must be selected together with receiving pressure, compressor power, turndown, liquid dropout, linepack, capacity booking and expansion strategy.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for length_km, group in gas_lateral_screen.groupby("length_km"):
    ax.plot(group.diameter_m * 1000.0, group.outlet_bara, marker="o", label=f"{length_km:.0f} km")
ax.axhline(120.0, color="black", linestyle="--", label="illustrative receiving pressure")
ax.set(xlabel="Inside diameter [mm]", ylabel="Outlet pressure [bara]", title="NeqSim gas-lateral screening")
ax.legend()
plt.show()

## Well-cost estimate

NeqSim's Norwegian regional estimator provides a repeatable screening basis. Results are not vendor quotes.

In [ ]:
producer_cost = WellCostEstimator(SubseaRegion.NORWAY)
producer_cost.calculateWellCost(
    "OIL_PRODUCER", "SEMI_SUBMERSIBLE", "SMART_COMPLETION",
    3600.0, 300.0, 75.0, 35.0, 0.0, True, 5
)
injector_cost = WellCostEstimator(SubseaRegion.NORWAY)
injector_cost.calculateWellCost(
    "GAS_INJECTOR", "SEMI_SUBMERSIBLE", "STANDARD",
    3400.0, 300.0, 65.0, 25.0, 0.0, True, 5
)
well_costs = pd.Series({
    "producer_each_musd": producer_cost.getTotalCost() / 1e6,
    "injector_each_musd": injector_cost.getTotalCost() / 1e6,
    "six_producers_musd": 6 * producer_cost.getTotalCost() / 1e6,
    "three_injectors_musd": 3 * injector_cost.getTotalCost() / 1e6,
})
display(well_costs.to_frame("value"))

## SURF-cost estimate

The reference SURF basis includes wells, infield flowlines, umbilical and risers. Route-specific concepts should each own a cost object.

In [ ]:
surf_estimator = SURFCostEstimator(9, 300.0, SubseaRegion.NORWAY)
surf_estimator.setInfieldFlowlineLengthKm(72.0)
surf_estimator.setInfieldFlowlineDiameterInches(12.0)
surf_estimator.setUmbilicalLengthKm(18.0)
surf_estimator.setRiserLengthM(450.0)
surf_estimator.setNumberOfProductionRisers(2)
surf_estimator.setExportPipelineLengthKm(0.0)
surf_cost_musd = surf_estimator.calculate() / 1e6
print("Reference SURF screening cost [MUSD]:", surf_cost_musd)

## SURF assurance register

Hydraulics and cost are necessary but not sufficient. A route must also pass flow assurance, mechanical design, geohazard, installation, inspection/repair and operability reviews.

In [ ]:
surf_register = pd.DataFrame([
    ["Steady-state pressure", "NeqSim pipe screening", "refine with well/network cases"],
    ["Hydrate", "not explicitly evaluated here", "cooldown/restart/inhibition"],
    ["Wax", "not explicitly evaluated here", "WAT/deposition/pigging"],
    ["Slugging", "steady-state indicator only", "dynamic assessment"],
    ["Mechanical integrity", "screening geometry", "code design and fatigue"],
    ["Route/geohazard", "public + synthetic map", "survey and routing study"],
    ["Crossings/installability", "not defined", "installation contractor input"],
], columns=["discipline", "current_evidence", "required_follow_up"])
display(surf_register)

## 8. Detailed greenfield facility

The greenfield factory is a real NeqSim process simulation: HP separation and water treatment, LP oil stabilization, export pumping, LP gas compression, gas recovery, export compression and two-stage injection compression with cooling.

In [ ]:
facility_strategy = configuration.getFacilityLifecycleStrategy()
design_preview = pd.Series({
    "strategy": str(facility_strategy.getName()),
    "development_mode": str(facility_strategy.getDevelopmentMode()),
    "design_margin": facility_strategy.getDesignMargin(),
    "auto_size_detailed_process": facility_strategy.isAutoSizeDetailedProcess(),
    "maximum_detailed_utilization": facility_strategy.getMaximumDetailedProcessUtilization(),
})
display(design_preview.to_frame("value"))

## Nameplate capacity basis

Nameplate envelopes cover oil, gas, water, liquid and power. Detailed equipment constraints can be more restrictive than aggregate nameplate.

In [ ]:
nameplate_preview = facility_strategy.getBaseCapacity()
nameplate_table = pd.Series({
    "oil_sm3_d": nameplate_preview.getOilSm3PerDay(),
    "gas_sm3_d": nameplate_preview.getGasSm3PerDay(),
    "water_sm3_d": nameplate_preview.getWaterSm3PerDay(),
    "liquid_sm3_d": nameplate_preview.getLiquidSm3PerDay(),
    "power_kw": nameplate_preview.getPowerKw(),
})
display(nameplate_table.to_frame("capacity"))

## Product specification object

The active specification object is part of the lifecycle configuration, so every annual process solve is checked consistently.

In [ ]:
specs = configuration.getProductSpecifications()
spec_table = pd.Series({
    "gas_CO2_max_mol_pct": specs.getMaximumGasCo2MolePercent(),
    "gas_H2S_max_ppm": specs.getMaximumGasH2sPpm(),
    "gas_O2_max_mol_pct": specs.getMaximumGasOxygenMolePercent(),
    "gas_GCV_min_MJ_Sm3": specs.getMinimumGasGrossCalorificValueMjPerSm3(),
    "gas_GCV_max_MJ_Sm3": specs.getMaximumGasGrossCalorificValueMjPerSm3(),
    "gas_Wobbe_min_MJ_Sm3": specs.getMinimumGasWobbeIndexMjPerSm3(),
    "gas_Wobbe_max_MJ_Sm3": specs.getMaximumGasWobbeIndexMjPerSm3(),
    "oil_RVP_max_bara": specs.getMaximumOilRvpBara(),
    "oil_BSW_max_vol_pct": specs.getMaximumOilBswVolumePercent(),
    "oil_in_water_max_mg_L": specs.getMaximumOilInWaterMgPerL(),
})
display(spec_table.to_frame("synthetic_limit"))

## Facility gaps for a project model

The factory is detailed enough to exercise NeqSim equipment and lifecycle coupling, but a project plant would add dehydration/dew-point control, produced-water polishing, fuel/flare, utilities, metering, chemical injection, safeguarding and realistic control/recycle convergence.

Creator functions should keep these additions standardized while allowing a full user ProcessSystem to replace them.

In [ ]:
facility_gap_register = pd.DataFrame([
    ["Oil stabilization", "HP/LP separation + pump", "add heat integration/column if required"],
    ["Gas compression", "LP/export/injection stages", "add scrubbers, antisurge and driver limits"],
    ["Gas quality", "live calculated checks", "add dehydration/dew-point unit"],
    ["Produced water", "treatment train", "calibrate performance and turndown"],
    ["Utilities", "power accounting", "build fuel/power/cooling medium systems"],
    ["Safety/control", "not a dynamic safety model", "add safeguarding and relief studies"],
], columns=["system", "reference_model", "project_upgrade"])
display(facility_gap_register)

## Run strategy

The complete area portfolio is evaluated once. Each option owns an independent reservoir and process state. This avoids state leakage and produces a consistent ranking.

In [ ]:
print("The integrated portfolio run is in the next cell.")
print("Expected work: four full reservoir-to-export lifecycle simulations.")

## 9. Integrated lifecycle portfolio run

This is the main computation. It evaluates standalone, host-priority, managed-host and remote-host routes, ranks eligible options by after-tax NPV, and retains full annual technical/economic results.

In [ ]:
area_evaluator = AreaDevelopmentEvaluator()
area_portfolio = NorwegianOilFieldCase.createAreaDevelopmentPortfolio()
area_result = area_evaluator.evaluate(area_portfolio)

display(Markdown(str(area_result.toMarkdownTable())))
print("Recommended eligible teaching option:",
      area_result.getRecommendedOption().getOption().getName()
      if area_result.getRecommendedOption() is not None else "none")

## Index results by route

Route metadata and lifecycle results are separated. This allows technical gates to be reported beside NPV.

In [ ]:
option_results = list(area_result.getRankedOptions())
results_by_option = {
    str(item.getOption().getName()): item.getLifecycleResult()
    for item in option_results
}
for name in results_by_option:
    print(name)

## Lifecycle summary helper

The summary includes both admitted and unconstrained/requested utilization. Requested utilization can exceed 100% and reveals the true bottleneck size.

In [ ]:
def lifecycle_summary(result):
    return {
        "concept": str(result.getConceptName()),
        "npv_musd": float(result.getNpvMusd()),
        "irr_pct": 100.0 * float(result.getIrr()),
        "payback_years": float(result.getPaybackYears()),
        "breakeven_oil_usd_bbl": float(result.getBreakevenOilPriceUsdPerBbl()),
        "oil_msm3": float(result.getCumulativeOilSm3()) / 1e6,
        "gas_export_gsm3": float(result.getCumulativeGasExportSm3()) / 1e9,
        "gas_injected_gsm3": float(result.getCumulativeGasInjectedSm3()) / 1e9,
        "water_msm3": float(result.getCumulativeWaterProducedSm3()) / 1e6,
        "deferred_oil_msm3": float(result.getCumulativeDeferredOilSm3()) / 1e6,
        "peak_admitted_util_pct": 100.0 * float(result.getPeakFacilityUtilization()),
        "peak_requested_util_pct": 100.0 * float(result.getPeakUnconstrainedFacilityUtilization()),
        "off_spec_years": int(result.getOffSpecificationYears()),
        "co2_kt": float(result.getLifecycleCo2Tonnes()) / 1000.0,
        "stop_reason": str(result.getStopReason()),
    }

portfolio_summary = pd.DataFrame([{
    "option": str(item.getOption().getName()),
    "route": str(item.getOption().getRouteType()),
    "receiving_asset": str(item.getOption().getReceivingAssetName()),
    "eligible": bool(item.isEligible()),
    **lifecycle_summary(item.getLifecycleResult())
} for item in option_results])
display(portfolio_summary)

## Annual-result helper

Annual records preserve production potential, requested and admitted rates, host load, holdback, deferment, bottlenecks, reservoir pressure, energy, emissions and specifications.

In [ ]:
def annual_frame(result):
    rows = []
    for item in result.getAnnualResults():
        quality = item.getProductSpecificationResult()
        rows.append({
            "year": int(item.getYear()),
            "oil_sm3": float(item.getOilSm3()),
            "oil_rate_sm3_d": float(item.getAverageOilRateSm3PerDay()),
            "potential_oil_sm3_d": float(item.getPotentialOilRateSm3PerDay()),
            "requested_oil_sm3_d": float(item.getRequestedOilRateSm3PerDay()),
            "host_oil_sm3_d": float(item.getHostOilRateSm3PerDay()),
            "host_gas_sm3_d": float(item.getHostGasRateSm3PerDay()),
            "host_water_sm3_d": float(item.getHostWaterRateSm3PerDay()),
            "gas_export_sm3": float(item.getGasExportSm3()),
            "gas_injected_sm3": float(item.getGasInjectedSm3()),
            "water_sm3": float(item.getWaterProducedSm3()),
            "water_cut": float(item.getAverageWaterCut()),
            "pressure_bara": float(item.getAverageReservoirPressureBara()),
            "holdback_oil_sm3": float(item.getHoldbackOilSm3()),
            "deferred_oil_sm3": float(item.getCapacityDeferredOilSm3()),
            "admitted_util": float(item.getMaximumFacilityUtilization()),
            "requested_util": float(item.getUnconstrainedFacilityUtilization()),
            "bottleneck": str(item.getPrimaryBottleneck()),
            "requested_bottleneck": str(item.getUnconstrainedBottleneck()),
            "energy_mwh": float(item.getEnergyMWh()),
            "co2_tonnes": float(item.getCo2EmissionsTonnes()),
            "gas_co2_mol_pct": float(quality.getGasCo2MolePercent()),
            "gas_h2s_ppm": float(quality.getGasH2sPpm()),
            "gas_gcv_mj_sm3": float(quality.getGasGrossCalorificValueMjPerSm3()),
            "gas_wobbe_mj_sm3": float(quality.getGasWobbeIndexMjPerSm3()),
            "oil_rvp_bara": float(quality.getOilRvpBara()),
            "oil_bsw_vol_pct": float(quality.getOilBswVolumePercent()),
            "oil_in_water_mg_l": float(quality.getOilInWaterMgPerL()),
            "quality_summary": str(quality.getSummary()),
        })
    return pd.DataFrame(rows)

annual_by_option = {name: annual_frame(result) for name, result in results_by_option.items()}
print({name: len(frame) for name, frame in annual_by_option.items()})

## Identify standalone and host cases

The area-option names are stable factory identifiers. The underlying concept names remain visible in the summary.

In [ ]:
standalone_name = next(name for name in results_by_option if "Standalone" in name)
host_a_priority_name = next(name for name in results_by_option if "Host A priority" in name)
host_a_managed_name = next(name for name in results_by_option if "Host A managed" in name)
host_b_name = next(name for name in results_by_option if "Remote host B" in name)

standalone = results_by_option[standalone_name]
host_a_priority = results_by_option[host_a_priority_name]
host_a_managed = results_by_option[host_a_managed_name]
host_b = results_by_option[host_b_name]
standalone_annual = annual_by_option[standalone_name]

## Greenfield facility design result

The facility is designed at a defined operating point before annual operation. Auto-sizing and mechanical constraints are retained in the result.

In [ ]:
design = standalone.getFacilityDesignResult()
capacity = design.getNameplateCapacity()
design_result = pd.Series({
    "strategy": str(design.getStrategyName()),
    "mode": str(design.getDevelopmentMode()),
    "design_oil_sm3_d": design.getDesignRates().getOilSm3PerDay(),
    "design_gas_sm3_d": design.getDesignRates().getGasSm3PerDay(),
    "design_water_sm3_d": design.getDesignRates().getWaterSm3PerDay(),
    "nameplate_oil_sm3_d": capacity.getOilSm3PerDay(),
    "nameplate_gas_sm3_d": capacity.getGasSm3PerDay(),
    "nameplate_water_sm3_d": capacity.getWaterSm3PerDay(),
    "design_power_kw": design.getDesignPowerKw(),
    "auto_sized_equipment_count": design.getAutoSizedEquipmentCount(),
    "mechanical_constraint_count": design.getMechanicalConstraintCount(),
    "parallel_train_count": design.getRequiredParallelTrainCount(),
    "design_bottleneck": str(design.getDesignBottleneck()),
    "design_bottleneck_util_pct": 100.0 * design.getDesignBottleneckUtilization(),
})
display(design_result.to_frame("value"))

## Standalone production and pressure

Gas injection supports pressure and production, while water breakthrough and facility constraints shape late life.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes[0, 0].plot(standalone_annual.year, standalone_annual.potential_oil_sm3_d, label="potential")
axes[0, 0].plot(standalone_annual.year, standalone_annual.oil_rate_sm3_d, label="produced")
axes[0, 0].set_ylabel("Oil [Sm3/d]"); axes[0, 0].legend()
axes[0, 1].plot(standalone_annual.year, standalone_annual.pressure_bara)
axes[0, 1].set_ylabel("Reservoir pressure [bara]")
axes[1, 0].plot(standalone_annual.year, standalone_annual.gas_export_sm3 / 1e6, label="export")
axes[1, 0].plot(standalone_annual.year, standalone_annual.gas_injected_sm3 / 1e6, label="injected")
axes[1, 0].set_ylabel("Annual gas [MSm3]"); axes[1, 0].legend()
axes[1, 1].plot(standalone_annual.year, 100.0 * standalone_annual.water_cut)
axes[1, 1].set_ylabel("Water cut [%]")
for ax in axes[1, :]: ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

## Requested versus admitted utilization

Admitted utilization is capped by allocation. Requested utilization exposes overload and should drive modification sizing.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(standalone_annual.year, 100.0 * standalone_annual.requested_util, label="requested")
ax.plot(standalone_annual.year, 100.0 * standalone_annual.admitted_util, label="admitted")
ax.axhline(100.0, color="black", linestyle="--")
ax.set(xlabel="Year", ylabel="Maximum utilization [%]", title="Standalone facility utilization")
ax.legend()
plt.show()

## Annual bottleneck register

The highest requested-utilization years identify the capacity constraint before production is reduced.

In [ ]:
standalone_bottlenecks = standalone_annual[[
    "year", "requested_bottleneck", "requested_util", "bottleneck",
    "admitted_util", "deferred_oil_sm3"
]].copy()
standalone_bottlenecks["requested_util_pct"] = 100.0 * standalone_bottlenecks.pop("requested_util")
standalone_bottlenecks["admitted_util_pct"] = 100.0 * standalone_bottlenecks.pop("admitted_util")
standalone_bottlenecks["deferred_oil_msm3"] = standalone_bottlenecks.pop("deferred_oil_sm3") / 1e6
display(standalone_bottlenecks.sort_values("requested_util_pct", ascending=False).head(15))

## Product quality through life

Live process streams are checked annually. Product quality can change as pressure, phase split, recycle and water cut change.

In [ ]:
quality_columns = [
    "year", "gas_co2_mol_pct", "gas_h2s_ppm", "gas_gcv_mj_sm3",
    "gas_wobbe_mj_sm3", "oil_rvp_bara", "oil_bsw_vol_pct",
    "oil_in_water_mg_l", "quality_summary"
]
display(standalone_annual[quality_columns])

## Quality visualization

Each axis should ultimately show the actual contract or permit limit. Here the dashed lines are the synthetic reference limits.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(standalone_annual.year, standalone_annual.gas_co2_mol_pct)
axes[0].axhline(specs.getMaximumGasCo2MolePercent(), color="red", linestyle="--")
axes[0].set_ylabel("Gas CO2 [mol%]")
axes[1].plot(standalone_annual.year, standalone_annual.oil_rvp_bara)
axes[1].axhline(specs.getMaximumOilRvpBara(), color="red", linestyle="--")
axes[1].set_ylabel("Oil RVP [bara]")
axes[2].plot(standalone_annual.year, standalone_annual.oil_in_water_mg_l)
axes[2].axhline(specs.getMaximumOilInWaterMgPerL(), color="red", linestyle="--")
axes[2].set_ylabel("Oil in water [mg/L]")
for ax in axes: ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

## Energy and emissions

The lifecycle records process energy and applies the configured grid emission factor. A real study must add fuel gas, flaring, venting, logistics, drilling and embodied emissions as required by the decision basis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(standalone_annual.year, standalone_annual.energy_mwh / 1000.0)
axes[0].set(ylabel="Energy [GWh/y]", xlabel="Year")
axes[1].bar(standalone_annual.year, standalone_annual.co2_tonnes / 1000.0)
axes[1].set(ylabel="CO2 [kt/y]", xlabel="Year")
plt.tight_layout()
plt.show()

## Cash-flow helper

The NeqSim result exposes taxes, depreciation, CAPEX, OPEX and discounted cash flow for every calendar year.

In [ ]:
def cashflow_frame(result):
    return pd.DataFrame([{
        "year": int(item.getYear()),
        "gross_revenue_musd": float(item.getGrossRevenue()),
        "tariff_musd": float(item.getTariff()),
        "capex_musd": float(item.getCapex()),
        "opex_musd": float(item.getOpex()),
        "depreciation_musd": float(item.getDepreciation()),
        "corporate_tax_musd": float(item.getCorporateTax()),
        "petroleum_tax_musd": float(item.getPetroleumTax()),
        "after_tax_cashflow_musd": float(item.getAfterTaxCashFlow()),
        "cumulative_cashflow_musd": float(item.getCumulativeCashFlow()),
        "discounted_cashflow_musd": float(item.getDiscountedCashFlow()),
    } for item in result.getCashFlowResult().getAnnualCashFlows()])

standalone_cashflow = cashflow_frame(standalone)
display(standalone_cashflow)

## Cash-flow visualization

CAPEX precedes first oil, and the after-tax profile determines payback and NPV. Break-even is solved on the same integrated schedule.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].bar(standalone_cashflow.year, standalone_cashflow.after_tax_cashflow_musd)
axes[0].set_ylabel("After-tax cash flow [MUSD]")
axes[1].plot(standalone_cashflow.year, standalone_cashflow.cumulative_cashflow_musd, marker="o")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set(ylabel="Cumulative cash flow [MUSD]", xlabel="Year")
plt.show()

## Standalone interpretation

The reference standalone case is technically integrated, but its economic outcome depends strongly on large greenfield CAPEX and the selected price basis. The result is a teaching outcome, not an NCS benchmark.

## 10. Brownfield host production

Brownfield results combine the satellite with time-varying host oil, gas and water. Host A and B use different synthetic decline/capacity envelopes.

In [ ]:
host_frames = {
    "Host A priority": annual_by_option[host_a_priority_name],
    "Host A managed": annual_by_option[host_a_managed_name],
    "Host B remote": annual_by_option[host_b_name],
}
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True)
for label, frame in host_frames.items():
    axes[0].plot(frame.year, frame.host_oil_sm3_d, label=label)
    axes[1].plot(frame.year, frame.host_gas_sm3_d / 1e6, label=label)
    axes[2].plot(frame.year, frame.host_water_sm3_d, label=label)
axes[0].set_ylabel("Host oil [Sm3/d]")
axes[1].set_ylabel("Host gas [MSm3/d]")
axes[2].set_ylabel("Host water [Sm3/d]")
for ax in axes: ax.set_xlabel("Year")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Host capacity utilization

Requested load shows the combined host and satellite demand. The host profile declines, so ullage generally grows, while water can remain restrictive.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label, frame in host_frames.items():
    ax.plot(frame.year, 100.0 * frame.requested_util, marker="o", label=label)
ax.axhline(100.0, color="black", linestyle="--")
ax.set(xlabel="Year", ylabel="Requested maximum utilization [%]", title="Synthetic host loading before allocation")
ax.legend()
plt.show()

## Holdback and deferred production

Holdback is a deliberate commercial/operational rate reduction. Capacity deferment is the additional oil rejected by constraints. They must be reported separately.

In [ ]:
holdback_rows = []
for label, frame in host_frames.items():
    holdback_rows.append({
        "case": label,
        "holdback_msm3": frame.holdback_oil_sm3.sum() / 1e6,
        "capacity_deferred_msm3": frame.deferred_oil_sm3.sum() / 1e6,
        "peak_requested_util_pct": 100.0 * frame.requested_util.max(),
    })
holdback_summary = pd.DataFrame(holdback_rows)
display(holdback_summary)

## Host-priority versus managed allocation

Host-priority protects base production and can defer the satellite heavily. Managed pro-rata allocation plus planned holdback can improve total value or utilization, but it requires agreement between owners.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label in ["Host A priority", "Host A managed"]:
    frame = host_frames[label]
    axes[0].plot(frame.year, frame.oil_rate_sm3_d, label=label)
    axes[1].plot(frame.year, frame.deferred_oil_sm3 / 1e6, label=label)
axes[0].set(ylabel="Satellite oil [Sm3/d]", xlabel="Year")
axes[1].set(ylabel="Deferred oil [MSm3/y]", xlabel="Year")
for ax in axes: ax.legend()
plt.tight_layout()
plt.show()

## Brownfield bottleneck frequency

Counting the requested bottleneck by year identifies recurring modification themes.

In [ ]:
bottleneck_frequency = []
for label, frame in host_frames.items():
    counts = frame.requested_bottleneck.value_counts()
    for bottleneck, count in counts.items():
        bottleneck_frequency.append({"case": label, "bottleneck": bottleneck, "years": count})
display(pd.DataFrame(bottleneck_frequency).sort_values(["case", "years"], ascending=[True, False]))

## Modification plan

The planner turns requested utilization into traceable screening candidates. SURF names lead to subsea actions; process names lead to equipment/train review.

In [ ]:
modification_plan = FacilityModificationPlanner().analyse(host_a_priority, 0.90)
display(Markdown(str(modification_plan.toMarkdownTable())))

## Modification interpretation

A capacity multiplier is not automatically an economic production credit. Build a cloned modified ProcessSystem/ProcessModel, add outage and CAPEX, rerun the lifecycle, then compare incremental NPV and feasibility.

In [ ]:
modification_candidates = pd.DataFrame([{
    "first_year": int(item.getFirstRequiredYear()),
    "bottleneck": str(item.getBottleneck()),
    "scope": str(item.getModificationScope()),
    "peak_requested_util_pct": 100.0 * float(item.getPeakRequestedUtilization()),
    "required_capacity_multiplier": float(item.getRequiredCapacityMultiplier()),
    "associated_deferred_oil_msm3": float(item.getAssociatedDeferredOilSm3()) / 1e6,
} for item in modification_plan.getCandidates()])
display(modification_candidates)

## Use an existing detailed ProcessSystem

The lifecycle model accepts the real brownfield NeqSim plant. Map reservoir, gas handling, export, host feeds and treated-water connections; the supplied process is executed at every admitted annual point.

```java
FieldLifecycleModel brownfield = FieldLifecycleModel
    .existingFacility("Aurora to real host", reservoir, existingHostProcessSystem)
    .reservoirStreams(oilProducer, waterProducer, gasInjector)
    .gasHandling(recoveredGas, allocationSplitter, compressedInjectionGas)
    .exportStreams(stabilizedOil, salesGas)
    .hostFeeds(hostOil, hostGas, hostWater)
    .treatedWaterDischarge(treatedWater)
    .productQualityProvider(realPlantAnalyser)
    .build();
```

## Use a multi-area ProcessModel

A multi-area model is useful when SURF, host topsides and export are owned or solved as separate process areas. The whole model is run, power is aggregated, and bottlenecks are reported as area::equipment.

```java
ProcessModel infrastructure = new ProcessModel();
infrastructure.add("shared SURF", existingSurfSystem);
infrastructure.add("host topsides", existingHostSystem);

FieldLifecycleModel route = FieldLifecycleModel
    .existingSurfAndFacility("Aurora via inherited route", reservoir,
        infrastructure, "shared SURF", "host topsides")
    .reservoirStreams(oilProducer, waterProducer, gasInjector)
    .gasHandling(recoveredGas, allocationSplitter, compressedInjectionGas)
    .exportStreams(stabilizedOil, salesGas)
    .hostFeeds(hostOil, hostGas, hostWater)
    .treatedWaterDischarge(treatedWater)
    .build();
```

## Existing SURF tie-in

If existing SURF and facility are already connected by shared NeqSim streams, the convenience overload assembles them into a two-area ProcessModel.

```java
FieldLifecycleModel.existingSurfAndFacility(
    "Aurora via shared manifold and riser",
    reservoir, existingSurfProcessSystem, existingFacilityProcessSystem);
```

This is the route for testing reuse of a manifold, flowline, riser or pipeline and determining whether the first modification belongs subsea or topsides.

## Brownfield connection-point checklist

Every connection must define pressure/temperature/composition basis, metering point, phase handling, host-production location, control ownership, backpressure propagation, shutdown alignment, turndown, product analyzers and constraint identifiers.

In [ ]:
brownfield_checklist = pd.DataFrame([
    ["Satellite wellstream", "mapped upstream of shared SURF/facility", False],
    ["Existing host feeds", "oil/gas/water mapped at real entry points", False],
    ["Oil export", "stabilized stream and metering point", False],
    ["Gas export/injection", "allocation and compressor outlets", False],
    ["Produced water", "treated discharge analyser", False],
    ["Equipment constraints", "mechanical and driver limits active", False],
    ["Host schedule", "validated annual/seasonal production", False],
    ["Commercial policy", "priority/pro-rata/holdback agreement", False],
], columns=["connection_or_gate", "requirement", "project_verified"])
display(brownfield_checklist)

## Brownfield result interpretation

The teaching results demonstrate mechanics: a lower-CAPEX tieback can have higher NPV, but host priority and water/gas constraints defer production. A real ranking may reverse after modifications, shutdowns, tariffs and access terms are added.

## 11. Area-development comparison

Area evaluation ranks only technically eligible options. Route, receiving asset, product compliance, NPV, break-even, oil recovery and capacity are visible together.

In [ ]:
area_comparison = portfolio_summary[[
    "option", "route", "receiving_asset", "eligible", "npv_musd",
    "breakeven_oil_usd_bbl", "oil_msm3", "deferred_oil_msm3",
    "peak_admitted_util_pct", "peak_requested_util_pct", "off_spec_years"
]].copy()
display(area_comparison)

## NPV and break-even comparison

NPV captures value at the selected assumptions; break-even indicates price resilience. Both remain conditional on feasibility.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=portfolio_summary, y="option", x="npv_musd", ax=axes[0], hue="eligible", dodge=False)
sns.barplot(data=portfolio_summary, y="option", x="breakeven_oil_usd_bbl", ax=axes[1], color="steelblue")
axes[0].set_title("After-tax NPV [MUSD]")
axes[1].set_title("Oil break-even [USD/bbl]")
plt.tight_layout()
plt.show()

## Recovery versus deferment

A high NPV option may recover less oil or defer more production. The decision should expose these trade-offs.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(portfolio_summary.deferred_oil_msm3, portfolio_summary.oil_msm3,
           s=np.maximum(portfolio_summary.npv_musd - portfolio_summary.npv_musd.min() + 100, 30),
           c=portfolio_summary.breakeven_oil_usd_bbl, cmap="viridis")
for _, row in portfolio_summary.iterrows():
    ax.annotate(row.option, (row.deferred_oil_msm3, row.oil_msm3), fontsize=8)
ax.set(xlabel="Deferred oil [MSm3]", ylabel="Produced oil [MSm3]",
       title="Recovery, deferment and value trade-off")
plt.show()

## Recompute economics consistently

The helper reconstructs NeqSim cash flow from a lifecycle production profile. It supports controlled price, CAPEX, OPEX and discount sensitivities without mutating the simulated reservoir/process result.

In [ ]:
def recompute_economics(result, configuration, oil_price=None, gas_price=None,
                          capex_factor=1.0, opex_factor=1.0, discount_rate=None):
    engine = CashFlowEngine()
    engine.setOpexPercentOfCapex(0.0)
    engine.setOilPrice(float(oil_price if oil_price is not None else configuration.getOilPriceUsdPerBbl()))
    engine.setGasPrice(float(gas_price if gas_price is not None else configuration.getGasPriceUsdPerSm3()))
    engine.setOilTariff(configuration.getOilTariffUsdPerBbl())
    engine.setGasTariff(configuration.getGasTariffUsdPerSm3())
    engine.setFixedOpexPerYear(configuration.getFixedOpexMusdPerYear() * opex_factor)
    engine.setFixedOpexStartYear(configuration.getStartYear())
    engine.setVariableOpexPerBoe(configuration.getVariableOpexUsdPerBoe() * opex_factor)
    for entry in configuration.getCapexScheduleMusd().entrySet():
        engine.addCapex(float(entry.getValue()) * capex_factor, int(entry.getKey()))
    for item in result.getAnnualResults():
        engine.addAnnualProduction(
            int(item.getYear()),
            float(item.getOilSm3()) * BARREL_PER_SM3,
            float(item.getGasExportSm3()),
            0.0
        )
    rate = float(discount_rate if discount_rate is not None else configuration.getDiscountRate())
    return engine.calculate(rate)

## Economic reconstruction check

The reconstructed base NPV should match the lifecycle result within numerical tolerance. This is an important validation before running sensitivities.

In [ ]:
standalone_config = NorwegianOilFieldCase.createGasInjectionCase().getConfiguration()
reconstructed = recompute_economics(standalone, standalone_config)
npv_check = pd.Series({
    "lifecycle_npv_musd": standalone.getNpvMusd(),
    "reconstructed_npv_musd": reconstructed.getNpv(),
    "difference_musd": reconstructed.getNpv() - standalone.getNpvMusd(),
})
display(npv_check.to_frame("value"))

## Oil-price sensitivity

Production is held constant in this post-processing sensitivity. A price-responsive economic limit would require rerunning the integrated lifecycle.

In [ ]:
oil_price_rows = []
for price in np.arange(40.0, 121.0, 5.0):
    result = recompute_economics(standalone, standalone_config, oil_price=price)
    oil_price_rows.append({"oil_price_usd_bbl": price, "npv_musd": result.getNpv()})
oil_price_sensitivity = pd.DataFrame(oil_price_rows)
display(oil_price_sensitivity)

## Oil-price curve and break-even

The zero crossing can be compared with NeqSim's direct break-even result.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(oil_price_sensitivity.oil_price_usd_bbl, oil_price_sensitivity.npv_musd, marker="o")
ax.axhline(0.0, color="black")
ax.axvline(standalone.getBreakevenOilPriceUsdPerBbl(), color="red", linestyle="--",
           label=f"NeqSim break-even {standalone.getBreakevenOilPriceUsdPerBbl():.1f}")
ax.set(xlabel="Oil price [USD/bbl]", ylabel="NPV [MUSD]")
ax.legend()
plt.show()

## CAPEX/OPEX sensitivity

Greenfield and tieback concepts have different exposure to CAPEX and host tariffs/OPEX. The matrix is recomputed on the selected standalone production profile.

In [ ]:
cost_rows = []
for capex_factor in [0.8, 1.0, 1.2, 1.4]:
    for opex_factor in [0.8, 1.0, 1.2, 1.4]:
        result = recompute_economics(
            standalone, standalone_config,
            capex_factor=capex_factor, opex_factor=opex_factor
        )
        cost_rows.append({
            "capex_factor": capex_factor, "opex_factor": opex_factor, "npv_musd": result.getNpv()
        })
cost_sensitivity = pd.DataFrame(cost_rows)
display(cost_sensitivity.pivot(index="opex_factor", columns="capex_factor", values="npv_musd"))

## Discount-rate sensitivity

Discount rate changes the value of delayed and deferred production, making long remote routes and late modifications especially sensitive.

In [ ]:
discount_rows = []
for rate in [0.04, 0.06, 0.08, 0.10, 0.12, 0.15]:
    result = recompute_economics(standalone, standalone_config, discount_rate=rate)
    discount_rows.append({"discount_rate_pct": 100.0 * rate, "npv_musd": result.getNpv()})
discount_sensitivity = pd.DataFrame(discount_rows)
display(discount_sensitivity)

## Injection-strategy comparison

The area portfolio contains natural depletion and the full injection case through its greenfield alternatives. A medium-recycle case adds a true integrated process/reservoir sensitivity.

In [ ]:
medium_injection_concept = NorwegianOilFieldCase.createCase(
    "Aurora Sør medium gas recycle", 0.50, 3.0e6
)
medium_injection_result = FieldLifecycleEvaluator().evaluate(medium_injection_concept)

natural_result = FieldLifecycleEvaluator().evaluate(
    NorwegianOilFieldCase.createNaturalDepletionCase()
)
injection_comparison = pd.DataFrame([
    {"strategy": "natural depletion", **lifecycle_summary(natural_result)},
    {"strategy": "50% recycle", **lifecycle_summary(medium_injection_result)},
    {"strategy": "85% recycle", **lifecycle_summary(standalone)},
])
display(injection_comparison[[
    "strategy", "npv_musd", "oil_msm3", "gas_export_gsm3",
    "gas_injected_gsm3", "breakeven_oil_usd_bbl", "co2_kt"
]])

## Injection trade-off

Gas injection can improve pressure support and oil recovery but reduces gas sales and adds compression, wells, power and CAPEX. The optimum is economic and reservoir-specific.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=injection_comparison, x="strategy", y="oil_msm3", ax=axes[0], color="steelblue")
sns.barplot(data=injection_comparison, x="strategy", y="npv_musd", ax=axes[1], color="seagreen")
sns.barplot(data=injection_comparison, x="strategy", y="co2_kt", ax=axes[2], color="darkorange")
axes[0].set_ylabel("Oil [MSm3]")
axes[1].set_ylabel("NPV [MUSD]")
axes[2].set_ylabel("CO2 [kt]")
for ax in axes: ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## Gas transport capacity scenarios

Physical export capacity, firm booked capacity and interruptible capacity are separate. The simple profile below shows how a commercial booking can be tighter than the process equipment.

In [ ]:
booking_scenarios = pd.DataFrame({
    "year": standalone_annual.year,
    "physical_capacity_msm3_d": 5.5,
    "firm_booked_msm3_d": np.where(standalone_annual.year < 2035, 2.0, 3.5),
    "firm_plus_interruptible_msm3_d": np.where(standalone_annual.year < 2035, 3.0, 4.5),
    "required_average_export_msm3_d": standalone_annual.gas_export_sm3 / DAYS_PER_YEAR / 1e6,
})
booking_scenarios["firm_shortfall_msm3_d"] = np.maximum(
    booking_scenarios.required_average_export_msm3_d - booking_scenarios.firm_booked_msm3_d, 0.0
)
display(booking_scenarios)

## Transport booking visualization

The profile is synthetic but implements the commercial distinction described on Gassco's booking page.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for column in ["physical_capacity_msm3_d", "firm_booked_msm3_d",
               "firm_plus_interruptible_msm3_d", "required_average_export_msm3_d"]:
    ax.plot(booking_scenarios.year, booking_scenarios[column], label=column.replace("_", " "))
ax.set(xlabel="Year", ylabel="Gas [MSm3/d]", title="Physical versus booked transport capacity")
ax.legend(fontsize=8)
plt.show()

## Post-processing uncertainty model

The Monte Carlo below varies oil price, gas price, CAPEX and OPEX around one integrated schedule. It is a conditional economic uncertainty, not a fully coupled probabilistic reservoir/process simulation.

In [ ]:
mc_rows = []
for _ in range(1000):
    oil_price = max(20.0, rng.normal(75.0, 15.0))
    gas_price = max(0.05, rng.normal(0.28, 0.06))
    capex_factor = rng.lognormal(mean=0.0, sigma=0.18)
    opex_factor = rng.lognormal(mean=0.0, sigma=0.12)
    econ = recompute_economics(
        standalone, standalone_config,
        oil_price=oil_price, gas_price=gas_price,
        capex_factor=capex_factor, opex_factor=opex_factor
    )
    mc_rows.append({
        "oil_price": oil_price, "gas_price": gas_price,
        "capex_factor": capex_factor, "opex_factor": opex_factor,
        "npv_musd": econ.getNpv()
    })
economic_mc = pd.DataFrame(mc_rows)
display(economic_mc.npv_musd.describe(percentiles=[0.1, 0.5, 0.9]))

## Economic uncertainty visualization

The probability of positive NPV is conditional on the stated distributions and fixed technical schedule.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(economic_mc.npv_musd, bins=40, kde=True, ax=axes[0])
axes[0].axvline(0.0, color="black")
axes[0].set_xlabel("NPV [MUSD]")
sns.scatterplot(data=economic_mc, x="oil_price", y="npv_musd", hue="capex_factor",
                palette="viridis", alpha=0.6, ax=axes[1], legend=False)
axes[1].set(xlabel="Oil price [USD/bbl]", ylabel="NPV [MUSD]")
plt.tight_layout()
plt.show()
print("Conditional probability NPV > 0:", (economic_mc.npv_musd > 0).mean())

## Decision matrix

A transparent decision matrix keeps non-economic evidence visible. Scores below are illustrative and must be replaced by a governed multidisciplinary assessment.

In [ ]:
decision_matrix = portfolio_summary[[
    "option", "npv_musd", "breakeven_oil_usd_bbl", "deferred_oil_msm3",
    "peak_requested_util_pct", "off_spec_years"
]].copy()
decision_matrix["economic_score"] = decision_matrix.npv_musd.rank(pct=True) * 100.0
decision_matrix["resilience_score"] = (1.0 - decision_matrix.breakeven_oil_usd_bbl.rank(pct=True)) * 100.0
decision_matrix["capacity_score"] = (1.0 - decision_matrix.deferred_oil_msm3.rank(pct=True)) * 100.0
decision_matrix["technical_maturity_score"] = decision_matrix.option.map({
    "Standalone FPSO": 55.0,
    "Host A priority tieback": 65.0,
    "Host A managed tieback": 55.0,
    "Remote host B tieback": 45.0,
}).fillna(50.0)
weights = {"economic_score": 0.40, "resilience_score": 0.20,
           "capacity_score": 0.20, "technical_maturity_score": 0.20}
decision_matrix["illustrative_weighted_score"] = sum(
    decision_matrix[column] * weight for column, weight in weights.items()
)
display(decision_matrix.sort_values("illustrative_weighted_score", ascending=False))

## Decision robustness

A concept is robust only if it remains feasible and competitive across subsurface, route, host, schedule, cost and market uncertainty. A single weighted score must never hide a failed gate.

In [ ]:
robustness_register = pd.DataFrame([
    ["Standalone", "independent host", "high CAPEX and execution exposure", "facility optimization"],
    ["Host A priority", "lower CAPEX", "satellite deferment and host access", "capacity/modification agreement"],
    ["Host A managed", "better sharing potential", "commercial complexity", "allocation agreement"],
    ["Host B remote", "alternative host", "long SURF and lower maturity", "route/hydraulic survey"],
], columns=["route", "strength", "principal_risk", "next_value_step"])
display(robustness_register)

## 12. Detailed well, SURF and process engineering

This section turns the concept comparison into an explicit design study. It sizes production and injection wells, screens tubing and subsea flowlines, records the SURF architecture, extracts detailed NeqSim equipment sizing, builds a declining-host capacity envelope and evaluates debottleneck options.

All dimensions remain screening results. Code design, fatigue, materials, erosion, flow assurance, vendor curves, availability, safeguarding and layout require dedicated discipline verification.

## Engineering design philosophy

Size at a defined peak/design envelope, then test the entire life for turndown and overload. The design point should include production uncertainty, host load, availability, operating modes, margins and simultaneous-operation rules.

For brownfield service, do not auto-size the inherited plant. Load the actual installed dimensions and mechanical/driver limits, then use NeqSim utilization constraints to identify overload.

In [ ]:
engineering_design_basis = pd.DataFrame([
    ["Production wells", "6", "plateau deliverability with one-well sensitivity"],
    ["Gas injectors", "3", "injectivity and fracture-pressure limits"],
    ["Production tubing", "candidate IDs 89-127 mm", "vertical multiphase pressure margin"],
    ["Production flowline", "12/35/65 km; candidate IDs 254-406 mm", "host tie-in pressure"],
    ["Risers", "2 production risers in cost basis", "velocity, redundancy and operability"],
    ["Manifold", "6 producers + 3 injectors + spare philosophy", "slot and isolation basis"],
    ["Umbilical", "18 km reference", "controls, chemicals, power/fibre"],
    ["Greenfield process", "HP/LP separation, water, export, injection", "auto-sized at design rate"],
    ["Brownfield host", "declining base production", "actual installed capacities, no auto-size"],
], columns=["system", "screening_basis", "design_objective"])
display(engineering_design_basis)

## Production-well transient model

TransientWellModel provides a line-source drawdown calculation with permeability, thickness, porosity, compressibility, viscosity, formation-volume factor, drainage radius and skin. It complements the lifecycle's transparent PI limit.

In [ ]:
TransientWellModel = JClass("neqsim.process.fielddevelopment.reservoir.TransientWellModel")
TransientWellType = JClass("neqsim.process.fielddevelopment.reservoir.TransientWellModel$WellType")
BoundaryType = JClass("neqsim.process.fielddevelopment.reservoir.TransientWellModel$BoundaryType")

production_well = (
    TransientWellModel()
    .setWellType(TransientWellType.OIL_PRODUCER)
    .setBoundaryType(BoundaryType.INFINITE)
    .setReservoirPressure(330.0, "bara")
    .setPermeability(180.0, "mD")
    .setFormationThickness(45.0, "m")
    .setPorosity(0.22)
    .setTotalCompressibility(1.5e-4, "1/bar")
    .setFluidViscosity(0.8, "cP")
    .setFormationVolumeFactor(1.25)
    .setWellboreRadius(0.108, "m")
    .setDrainageRadius(650.0, "m")
    .setSkinFactor(2.0)
)
print("Hydraulic diffusivity [m2/s]:", production_well.getHydraulicDiffusivity())
print("Transmissibility [m3/(bar s)]:", production_well.getTransmissibility())

## Per-well drawdown envelope

The plateau target is about 3,833 Sm3/d per producer. Drawdown is checked from one day to one year to expose transient and boundary sensitivity.

In [ ]:
well_drawdown_rows = []
for time_hours in [24.0, 168.0, 720.0, 8760.0]:
    for rate_sm3_d in [2000.0, 3000.0, 3833.0, 4500.0, 5500.0, 6500.0]:
        result = production_well.calculateDrawdown(rate_sm3_d, time_hours)
        well_drawdown_rows.append({
            "time_hours": time_hours,
            "rate_sm3_d": rate_sm3_d,
            "flowing_bhp_bara": float(result.flowingPressure),
            "drawdown_bar": float(result.drawdown),
            "pi_sm3_d_bar": float(result.productivityIndex),
            "radius_investigation_m": float(result.radiusOfInvestigation),
            "infinite_acting": bool(result.infiniteActing),
        })
well_drawdown = pd.DataFrame(well_drawdown_rows)
display(well_drawdown)

## Drawdown acceptance

A production target must retain pressure for tubing, choke and SURF losses and stay above sand, coning, completion and lift limits. The notebook uses 250 bara as the lifecycle minimum BHP, which is deliberately conservative for the integrated example.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for time_hours, group in well_drawdown.groupby("time_hours"):
    ax.plot(group.rate_sm3_d, group.flowing_bhp_bara, marker="o", label=f"{time_hours:g} h")
ax.axhline(configuration.getMinimumBottomHolePressureBara(), color="red", linestyle="--",
           label="lifecycle minimum BHP")
ax.set(xlabel="Per-well oil rate [Sm3/d]", ylabel="Flowing BHP [bara]",
       title="Transient production-well drawdown")
ax.legend()
plt.show()

## Skin and permeability sensitivity

Completion damage and reservoir quality determine well count and stimulation value. This grid tests the plateau per-well target after 30 days.

In [ ]:
well_sensitivity_rows = []
for permeability_md in [50.0, 100.0, 180.0, 300.0]:
    for skin in [0.0, 2.0, 5.0, 10.0]:
        test = (
            TransientWellModel().setWellType(TransientWellType.OIL_PRODUCER)
            .setReservoirPressure(330.0, "bara").setPermeability(permeability_md, "mD")
            .setFormationThickness(45.0, "m").setPorosity(0.22)
            .setTotalCompressibility(1.5e-4, "1/bar").setFluidViscosity(0.8, "cP")
            .setFormationVolumeFactor(1.25).setWellboreRadius(0.108, "m")
            .setDrainageRadius(650.0, "m").setSkinFactor(skin)
        )
        result = test.calculateDrawdown(3833.0, 720.0)
        well_sensitivity_rows.append({
            "permeability_md": permeability_md, "skin": skin,
            "flowing_bhp_bara": float(result.flowingPressure),
            "drawdown_bar": float(result.drawdown)
        })
well_sensitivity = pd.DataFrame(well_sensitivity_rows)
display(well_sensitivity.pivot(index="skin", columns="permeability_md", values="flowing_bhp_bara"))

## Production tubing diameter screening

A single-well NeqSim multiphase pipe is run vertically across candidate tubing IDs. The feed rate is a screening mass-rate representation of the compositional wellstream; a project model should use well-specific PVT and phase rates.

In [ ]:
def simulate_production_tubing(tubing_id_m, mass_rate_kg_s, inlet_bara=330.0):
    fluid = NorwegianOilFieldCase.createReservoirFluid()
    fluid.setPressure(float(inlet_bara), "bara")
    fluid.setTemperature(90.0, "C")
    ThermodynamicOperations(fluid).TPflash()
    inlet = Stream(f"producer tubing feed {tubing_id_m}", fluid)
    inlet.setFlowRate(float(mass_rate_kg_s), "kg/sec")
    inlet.run()
    pipe = PipeBeggsAndBrills(f"producer tubing {tubing_id_m}", inlet)
    pipe.setLength(2800.0)
    pipe.setElevation(2200.0)
    pipe.setDiameter(float(tubing_id_m))
    pipe.setPipeWallRoughness(2.5e-5)
    pipe.setNumberOfIncrements(30)
    pipe.run()
    return {
        "wellhead_bara": float(pipe.getOutletStream().getPressure("bara")),
        "wellhead_temperature_c": float(pipe.getOutletStream().getTemperature("C")),
    }

tubing_rows = []
for rate_kg_s in [30.0, 40.0, 50.0, 60.0]:
    for tubing_id_m in [0.0889, 0.1016, 0.1143, 0.1270]:
        try:
            result = simulate_production_tubing(tubing_id_m, rate_kg_s)
            tubing_rows.append({"mass_rate_kg_s": rate_kg_s, "tubing_id_m": tubing_id_m, **result, "status": "ok"})
        except Exception as error:
            tubing_rows.append({"mass_rate_kg_s": rate_kg_s, "tubing_id_m": tubing_id_m,
                                "wellhead_bara": np.nan, "wellhead_temperature_c": np.nan,
                                "status": str(error)[:100]})
tubing_screen = pd.DataFrame(tubing_rows)
display(tubing_screen)

## Tubing selection plot

Larger tubing reduces friction but can worsen liquid lifting and turndown. The selected ID must be checked for erosion, buckling, completion compatibility, gas lift and late-life stability.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for rate, group in tubing_screen.groupby("mass_rate_kg_s"):
    ax.plot(group.tubing_id_m * 1000.0, group.wellhead_bara, marker="o", label=f"{rate:.0f} kg/s")
ax.set(xlabel="Tubing ID [mm]", ylabel="Predicted wellhead pressure [bara]",
       title="NeqSim vertical tubing screening")
ax.legend(title="Wellstream rate")
plt.show()

## Producer-count sizing

Well count is the maximum of plateau-rate demand, one-well-out availability and subsurface drainage requirements. Only the first two are quantified here.

In [ ]:
target_plateau = configuration.getPlateauOilRateSm3PerDay()
per_well_target = 3833.0
rate_based_wells = math.ceil(target_plateau / per_well_target)
one_well_out_wells = math.ceil(target_plateau / per_well_target) + 1
well_count_options = pd.DataFrame([
    ["Base six producers", 6, 6 * per_well_target, False, "reference case"],
    ["One-well-out plateau", one_well_out_wells, (one_well_out_wells - 1) * per_well_target, True,
     "higher CAPEX and resilience"],
    ["Five producers", 5, 5 * per_well_target, False, "cannot sustain target at assumed per-well rate"],
], columns=["option", "drilled_producers", "available_plateau_sm3_d", "one_well_out_basis", "comment"])
well_count_options["meets_plateau"] = well_count_options.available_plateau_sm3_d >= target_plateau
display(well_count_options)

## Well engineering deliverables

A complete well concept adds trajectory, casing/tubing stress, completion, sand control, inflow control, artificial lift, wellhead/tree, barrier diagrams, materials/corrosion, intervention, drilling schedule, reliability and abandonment.

In [ ]:
well_deliverables = pd.DataFrame([
    ["Subsurface target/trajectory", "not defined", "3D reservoir and collision study"],
    ["Inflow and transient response", "NeqSim screening", "well test / nodal calibration"],
    ["Tubing hydraulics", "NeqSim multiphase screening", "thermal and lift optimization"],
    ["Completion/sand control", "smart-completion cost assumption", "formation-specific design"],
    ["Tree/wellhead", "subsea assumption", "pressure/material class"],
    ["Integrity/barriers", "not designed", "NORSOK/API well design"],
    ["Intervention/abandonment", "not costed in detail", "lifecycle strategy"],
], columns=["deliverable", "notebook_status", "required_next_step"])
display(well_deliverables)

## SURF architecture sizing

The selected architecture contains production trees, injection trees, a manifold, flowlines, risers and umbilical. Spare slots and retrievability are explicit design choices, not generic percentage margins.

In [ ]:
surf_architecture = pd.DataFrame([
    ["Production trees", 6, "10k/15k class to be selected", "well isolation and monitoring"],
    ["Gas-injection trees", 3, "injection/fracture pressure basis", "non-return and integrity"],
    ["Manifold production slots", 8, "6 active + 2 spare", "future/infill flexibility"],
    ["Manifold injection slots", 4, "3 active + 1 spare", "injection flexibility"],
    ["Production flowlines", 2, "looped/dual route screening", "capacity and pigging"],
    ["Production risers", 2, "450 m cost basis", "availability and slug handling"],
    ["Injection line/riser", 1, "350 bara basis", "fracture/integrity control"],
    ["Umbilical", 1, "18 km reference", "hydraulic/electric/fibre/chemicals"],
], columns=["item", "count", "sizing_basis", "principal_function"])
display(surf_architecture)

## Flowline selection criteria

The hydraulic table already compares 12, 35 and 65 km routes. This cell applies a screening receiving-pressure gate and flags practical candidates.

In [ ]:
flowline_selection = multiphase_screen.copy()
flowline_selection["receiving_pressure_margin_bar"] = flowline_selection.outlet_bara - 45.0
flowline_selection["hydraulic_pass"] = flowline_selection.receiving_pressure_margin_bar >= 10.0
flowline_selection["nominal_inch_approx"] = flowline_selection.diameter_m / 0.0254
display(flowline_selection.sort_values(["length_km", "diameter_m"]))

## Flowline design selection

Hydraulic pass is only one gate. Very large diameter can increase liquid holdup and cooldown inventory, while small diameter increases friction and erosion. Route-specific thermal insulation and transient analysis are required.

In [ ]:
preferred_flowlines = (
    flowline_selection[flowline_selection.hydraulic_pass]
    .sort_values(["length_km", "diameter_m"])
    .groupby("length_km", as_index=False)
    .first()
)
display(preferred_flowlines[[
    "length_km", "diameter_m", "nominal_inch_approx",
    "outlet_bara", "receiving_pressure_margin_bar"
]])

## Riser velocity screening

A velocity estimate is constructed from design liquid and compressed-gas volume. It is intentionally conservative and used only to compare candidate riser IDs.

In [ ]:
design_oil_sm3_d = design.getDesignRates().getOilSm3PerDay()
design_water_sm3_d = design.getDesignRates().getWaterSm3PerDay()
design_gas_sm3_d = design.getDesignRates().getGasSm3PerDay()
# Approximate gas volume at 55 bara and 50 C from standard conditions.
gas_actual_m3_d = design_gas_sm3_d * (1.01325 / 55.0) * ((273.15 + 50.0) / 288.15)
total_actual_m3_s = (design_oil_sm3_d + design_water_sm3_d + gas_actual_m3_d) / 86400.0
riser_rows = []
for riser_id_m in [0.254, 0.305, 0.356, 0.406]:
    area_m2 = math.pi * riser_id_m ** 2 / 4.0
    velocity_m_s = total_actual_m3_s / (2.0 * area_m2)
    riser_rows.append({
        "riser_id_m": riser_id_m, "parallel_risers": 2,
        "approx_velocity_m_s": velocity_m_s,
        "screening_velocity_pass": 1.0 <= velocity_m_s <= 15.0
    })
riser_screen = pd.DataFrame(riser_rows)
display(riser_screen)

## Umbilical and chemical-service basis

Umbilical sizing is function-based: valve functions and accumulators, chemical rates and pressure, electric load, fibre count, redundancy, step-out voltage drop and dynamic/static sections.

In [ ]:
umbilical_basis = pd.DataFrame([
    ["Hydraulic control", "trees/manifold valves", "simultaneous operations and response time"],
    ["MEG/thermodynamic inhibitor", "startup/restart basis", "flow-assurance study"],
    ["Wax inhibitor", "if required by WAT/deposition", "PVT/flow-assurance data"],
    ["Scale inhibitor", "brine compatibility", "formation/produced-water chemistry"],
    ["Corrosion inhibitor", "materials and water wetting", "corrosion model"],
    ["Power", "subsea controls/instrumentation", "voltage drop and redundancy"],
    ["Fibre", "control and monitoring", "channel count and spare fibres"],
], columns=["service", "sizing_driver", "missing_evidence"])
display(umbilical_basis)

## SURF cost sensitivity

Well count and route length are varied using independent NeqSim SURFCostEstimator objects. This isolates the main architecture cost drivers.

In [ ]:
surf_cost_rows = []
for well_count in [6, 9, 11]:
    for infield_length_km in [35.0, 72.0, 120.0]:
        estimator = SURFCostEstimator(well_count, 300.0, SubseaRegion.NORWAY)
        estimator.setInfieldFlowlineLengthKm(infield_length_km)
        estimator.setInfieldFlowlineDiameterInches(12.0)
        estimator.setUmbilicalLengthKm(max(18.0, infield_length_km / 4.0))
        estimator.setRiserLengthM(450.0)
        estimator.setNumberOfProductionRisers(2)
        estimator.setExportPipelineLengthKm(0.0)
        surf_cost_rows.append({
            "well_count": well_count, "infield_length_km": infield_length_km,
            "surf_cost_musd": estimator.calculate() / 1e6
        })
surf_cost_sensitivity = pd.DataFrame(surf_cost_rows)
display(surf_cost_sensitivity)

## SURF cost response

The estimator is a Class-4 screening tool. Installation spread, market, crossings, rock dumping, trenching, procurement, contingency and schedule risk require explicit project treatment.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for count, group in surf_cost_sensitivity.groupby("well_count"):
    ax.plot(group.infield_length_km, group.surf_cost_musd, marker="o", label=f"{count} wells")
ax.set(xlabel="Total infield flowline length [km]", ylabel="SURF cost [MUSD]",
       title="NeqSim SURF screening-cost sensitivity")
ax.legend()
plt.show()

## Detailed process sizing extraction

The area evaluator mutated each independent concept through its full lifecycle. Its greenfield model retains the auto-sized mechanical design. The following cells inspect the same ProcessSystem used in the standalone result.

In [ ]:
standalone_option_result = next(
    item for item in option_results if str(item.getOption().getName()) == standalone_name
)
standalone_process = (
    standalone_option_result.getOption().getLifecycleConcept().getModel().getProcessSystem()
)
print("Detailed process:", standalone_process.getName())
print("Units:", len(standalone_process.getUnitOperations()))

## NeqSim design report

ProcessSystem.getDesignReportJson returns auto-sized equipment, dimensions and registered capacity constraints. This provides a machine-readable engineering handoff.

In [ ]:
design_report = json.loads(str(standalone_process.getDesignReportJson()))
design_equipment = design_report.get("equipment", [])
print("Auto-sizeable equipment records:", len(design_equipment))
display(pd.DataFrame([{
    "name": item.get("name"),
    "type": item.get("type"),
    "auto_sized": item.get("autoSized"),
    "max_util_pct": item.get("capacityConstraints", {}).get("maxUtilization", np.nan),
    "capacity_exceeded": item.get("capacityConstraints", {}).get("capacityExceeded", False),
} for item in design_equipment]))

## Flatten equipment sizing data

Sizing reports vary by equipment type. json_normalize preserves separator dimensions, compressor/pump design values and constraint metadata without forcing a single schema.

In [ ]:
sizing_rows = []
for item in design_equipment:
    base = {"name": item.get("name"), "type": item.get("type"), "auto_sized": item.get("autoSized")}
    sizing = item.get("sizingData", {})
    flat = pd.json_normalize(sizing, sep=".").to_dict(orient="records")
    sizing_rows.append({**base, **(flat[0] if flat else {})})
equipment_sizing_flat = pd.DataFrame(sizing_rows)
display(equipment_sizing_flat)

## Separator dimensions

HP and LP separators are sized from the greenfield design point. Mechanical-design rules and internals must be confirmed for gas capacity, oil/water retention, surge, slug volume and nozzle momentum.

In [ ]:
separator_rows = []
for separator_name in ["HP separator", "LP separator"]:
    separator = standalone_process.getUnit(separator_name)
    separator_rows.append({
        "name": separator_name,
        "internal_diameter_m": float(separator.getInternalDiameter()),
        "tangent_length_m": float(separator.getSeparatorLength()),
        "capacity_duty": float(separator.getCapacityDuty()),
        "capacity_max": float(separator.getCapacityMax()),
        "utilization_pct": 100.0 * float(separator.getCapacityDuty() / separator.getCapacityMax())
            if separator.getCapacityMax() > 0 else np.nan,
    })
separator_design = pd.DataFrame(separator_rows)
display(separator_design)

## Compressor and pump power

Driver sizing must include ambient derating, degradation, start-up, antisurge recycle, settle-out, motor/VSD efficiency and spare philosophy. The reported values are process shaft requirements.

In [ ]:
power_rows = []
for unit in standalone_process.getUnitOperations():
    class_name = str(unit.getClass().getSimpleName())
    if class_name == "Compressor":
        power_rows.append({"name": str(unit.getName()), "type": class_name,
                           "shaft_power_kw": float(unit.getPower("kW"))})
    elif class_name == "Pump":
        power_rows.append({"name": str(unit.getName()), "type": class_name,
                           "shaft_power_kw": float(unit.getPower("kW"))})
power_breakdown = pd.DataFrame(power_rows)
display(power_breakdown.sort_values("shaft_power_kw", ascending=False))

## Power train sizing

A screening train count is calculated at 85% maximum continuous driver loading with one installed spare only where stated. This complements the lifecycle's process-wide train indication.

In [ ]:
DRIVER_RATING_KW = 15000.0
MAX_CONTINUOUS_FRACTION = 0.85
power_train_rows = []
for _, row in power_breakdown.iterrows():
    running_trains = max(1, math.ceil(row.shaft_power_kw / (DRIVER_RATING_KW * MAX_CONTINUOUS_FRACTION)))
    spare_trains = 1 if "export" in row["name"].lower() or "injection" in row["name"].lower() else 0
    power_train_rows.append({
        **row.to_dict(),
        "driver_rating_kw": DRIVER_RATING_KW,
        "running_trains": running_trains,
        "installed_spares": spare_trains,
        "installed_trains": running_trains + spare_trains,
    })
power_train_sizing = pd.DataFrame(power_train_rows)
display(power_train_sizing)

## Cooler-duty screening

Cooler duties define cooling-medium or air-cooler area and utility load. Sign conventions are normalized to absolute MW.

In [ ]:
cooler_rows = []
for unit in standalone_process.getUnitOperations():
    if str(unit.getClass().getSimpleName()) == "Cooler":
        cooler_rows.append({
            "name": str(unit.getName()),
            "absolute_duty_mw": abs(float(unit.getDuty())) / 1e6
        })
cooler_duties = pd.DataFrame(cooler_rows)
display(cooler_duties.sort_values("absolute_duty_mw", ascending=False))

## Mechanical capacity constraints

The utilization snapshot lists every enabled constraint, current/design values, feasibility and hard-limit status. It is the detailed basis for bottleneck review.

In [ ]:
utilization_snapshot = json.loads(str(standalone_process.getUtilizationSnapshotJson()))
constraint_rows = []
for unit in utilization_snapshot.get("units", []):
    constraints = unit.get("constraints", [])
    if constraints:
        for constraint in constraints:
            constraint_rows.append({
                "equipment": unit.get("name"),
                "type": unit.get("type"),
                "constraint": constraint.get("name"),
                "current": constraint.get("current"),
                "design": constraint.get("design"),
                "unit": constraint.get("unit"),
                "utilization_pct": constraint.get("utilizationPercent"),
                "violated": constraint.get("violated"),
                "data_source": constraint.get("dataSource"),
            })
    else:
        constraint_rows.append({
            "equipment": unit.get("name"), "type": unit.get("type"),
            "constraint": None, "utilization_pct": unit.get("maxUtilizationPercent")
        })
detailed_constraints = pd.DataFrame(constraint_rows)
display(detailed_constraints.sort_values("utilization_pct", ascending=False).head(30))

## Facility sizing summary

The facility must be sized by dimension, not by one oil-rate number. Gas, water, liquid, power and individual equipment can each control in a different year.

In [ ]:
facility_dimension_summary = pd.DataFrame([
    ["Oil", capacity.getOilSm3PerDay(), "Sm3/d", "separation/stabilization/export"],
    ["Gas", capacity.getGasSm3PerDay(), "Sm3/d", "compression/export/injection"],
    ["Water", capacity.getWaterSm3PerDay(), "Sm3/d", "separation/treatment/disposal"],
    ["Liquid", capacity.getLiquidSm3PerDay(), "Sm3/d", "inlet/separators"],
    ["Power", design.getDesignPowerKw(), "kW", "drivers/electrical system"],
], columns=["dimension", "design_or_nameplate", "unit", "main_systems"])
display(facility_dimension_summary)

## Facility utilities and support systems

A complete design must close power generation/import, fuel gas, cooling, heating, flare, instrument air, nitrogen, chemicals, water systems, drainage, HVAC and emergency systems. Their loads scale differently with production.

In [ ]:
utility_register = pd.DataFrame([
    ["Electrical power", design.getDesignPowerKw(), "kW", "main compression/pumping + support margin"],
    ["Cooling", cooler_duties.absolute_duty_mw.sum() if not cooler_duties.empty else np.nan, "MW", "cooler duty sum"],
    ["Heating", np.nan, "MW", "oil stabilization/dehydration basis required"],
    ["Fuel gas", np.nan, "Sm3/d", "driver concept dependent"],
    ["Flare/relief", np.nan, "kg/s", "relief and depressurization study"],
    ["Chemical injection", np.nan, "L/h", "flow assurance/water chemistry"],
    ["Produced water", capacity.getWaterSm3PerDay(), "Sm3/d", "treatment nameplate"],
], columns=["utility", "screening_load", "unit", "basis"])
display(utility_register)

## Declining existing-host model

Host A is a mature producing field. Its synthetic base production declines, releasing oil and gas ullage, while water load can remain high. The satellite competes for the same inlet, separators, compression, water treatment, export and power.

In [ ]:
host_a_option_result = next(
    item for item in option_results if str(item.getOption().getName()) == host_a_priority_name
)
host_a_strategy = host_a_option_result.getOption().getLifecycleConcept().getConfiguration().getFacilityLifecycleStrategy()
host_a_facility = host_a_strategy.getHostFacility()

host_a_capacity = pd.Series({
    "oil_capacity_bopd": host_a_facility.getOilCapacityBopd(),
    "oil_capacity_sm3_d": host_a_facility.getOilCapacityBopd() / BARREL_PER_SM3,
    "gas_capacity_msm3_d": host_a_facility.getGasCapacityMSm3d(),
    "water_capacity_m3_d": host_a_facility.getWaterCapacityM3d(),
    "liquid_capacity_m3_d": host_a_facility.getLiquidCapacityM3d(),
    "synthetic_power_capacity_mw": 45.0,
})
display(host_a_capacity.to_frame("value"))

## Host ullage by dimension

Available capacity is calculated from the declining base profile before admitting Aurora Sør. Positive ullage is necessary but not sufficient because detailed equipment and process compatibility still apply.

In [ ]:
host_decline = host_frames["Host A priority"].copy()
host_decline["oil_capacity_sm3_d"] = host_a_capacity["oil_capacity_sm3_d"]
host_decline["gas_capacity_sm3_d"] = host_a_capacity["gas_capacity_msm3_d"] * 1e6
host_decline["water_capacity_sm3_d"] = host_a_capacity["water_capacity_m3_d"]
host_decline["liquid_capacity_sm3_d"] = host_a_capacity["liquid_capacity_m3_d"]
host_decline["oil_ullage_sm3_d"] = host_decline.oil_capacity_sm3_d - host_decline.host_oil_sm3_d
host_decline["gas_ullage_sm3_d"] = host_decline.gas_capacity_sm3_d - host_decline.host_gas_sm3_d
host_decline["water_ullage_sm3_d"] = host_decline.water_capacity_sm3_d - host_decline.host_water_sm3_d
host_decline["host_liquid_sm3_d"] = host_decline.host_oil_sm3_d + host_decline.host_water_sm3_d
host_decline["liquid_ullage_sm3_d"] = host_decline.liquid_capacity_sm3_d - host_decline.host_liquid_sm3_d
display(host_decline[[
    "year", "oil_ullage_sm3_d", "gas_ullage_sm3_d",
    "water_ullage_sm3_d", "liquid_ullage_sm3_d"
]])

## Host ullage visualization

Different dimensions release capacity at different rates. A single utilization number can obscure the reason for deferment.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes[0, 0].plot(host_decline.year, host_decline.oil_ullage_sm3_d)
axes[0, 0].set_ylabel("Oil ullage [Sm3/d]")
axes[0, 1].plot(host_decline.year, host_decline.gas_ullage_sm3_d / 1e6)
axes[0, 1].set_ylabel("Gas ullage [MSm3/d]")
axes[1, 0].plot(host_decline.year, host_decline.water_ullage_sm3_d)
axes[1, 0].set_ylabel("Water ullage [Sm3/d]")
axes[1, 1].plot(host_decline.year, host_decline.liquid_ullage_sm3_d)
axes[1, 1].set_ylabel("Liquid ullage [Sm3/d]")
for ax in axes[1, :]: ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

## Combined host and satellite loads

Satellite water and gas rates are reconstructed from annual volumes and availability. This gives an independent dimensional check beside NeqSim's full allocation result.

In [ ]:
operating_days = DAYS_PER_YEAR * standalone_config.getAvailability()
host_decline["satellite_requested_oil_sm3_d"] = host_decline.requested_oil_sm3_d
host_decline["satellite_water_sm3_d"] = host_decline.water_sm3 / operating_days
host_decline["satellite_gas_sm3_d"] = host_decline.gas_export_sm3 / operating_days
host_decline["combined_oil_util_pct"] = 100.0 * (
    host_decline.host_oil_sm3_d + host_decline.satellite_requested_oil_sm3_d
) / host_decline.oil_capacity_sm3_d
host_decline["combined_gas_util_pct"] = 100.0 * (
    host_decline.host_gas_sm3_d + host_decline.satellite_gas_sm3_d
) / host_decline.gas_capacity_sm3_d
host_decline["combined_water_util_pct"] = 100.0 * (
    host_decline.host_water_sm3_d + host_decline.satellite_water_sm3_d
) / host_decline.water_capacity_sm3_d
host_decline["combined_liquid_util_pct"] = 100.0 * (
    host_decline.host_liquid_sm3_d + host_decline.satellite_requested_oil_sm3_d
    + host_decline.satellite_water_sm3_d
) / host_decline.liquid_capacity_sm3_d
display(host_decline[[
    "year", "combined_oil_util_pct", "combined_gas_util_pct",
    "combined_water_util_pct", "combined_liquid_util_pct",
    "requested_util", "requested_bottleneck"
]])

## Dimension controlling the host

The independent maximum identifies whether oil, gas, water or total liquid is most restrictive each year. The detailed NeqSim bottleneck remains authoritative when equipment constraints are enabled.

In [ ]:
dimension_columns = {
    "oil": "combined_oil_util_pct",
    "gas": "combined_gas_util_pct",
    "water": "combined_water_util_pct",
    "liquid": "combined_liquid_util_pct",
}
host_decline["screening_controlling_dimension"] = host_decline[
    list(dimension_columns.values())
].idxmax(axis=1).map({value: key for key, value in dimension_columns.items()})
host_decline["screening_peak_dimension_util_pct"] = host_decline[
    list(dimension_columns.values())
].max(axis=1)
display(host_decline[[
    "year", "screening_controlling_dimension",
    "screening_peak_dimension_util_pct", "requested_bottleneck"
]])

## Synthetic host power capacity

Brownfield power is often a hidden constraint. A transparent load correlation represents base separation, compression, water handling and oil export. Replace it with the real host ProcessModel and driver/electrical limits.

In [ ]:
HOST_POWER_CAPACITY_MW = 45.0
host_decline["base_host_power_mw"] = (
    10.0
    + 20.0 * host_decline.host_gas_sm3_d / host_decline.gas_capacity_sm3_d
    + 6.0 * host_decline.host_water_sm3_d / host_decline.water_capacity_sm3_d
    + 5.0 * host_decline.host_oil_sm3_d / host_decline.oil_capacity_sm3_d
)
host_decline["satellite_process_power_mw"] = host_decline.energy_mwh / (DAYS_PER_YEAR * 24.0) / 1000.0
host_decline["combined_power_util_pct"] = 100.0 * (
    host_decline.base_host_power_mw + host_decline.satellite_process_power_mw
) / HOST_POWER_CAPACITY_MW
display(host_decline[[
    "year", "base_host_power_mw", "satellite_process_power_mw", "combined_power_util_pct"
]])

## Host bottleneck dashboard

Oil, gas, water, liquid, power and detailed requested utilization are shown together. This is the minimum view for a tie-in decision.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for column, label in [
    ("combined_oil_util_pct", "oil"),
    ("combined_gas_util_pct", "gas"),
    ("combined_water_util_pct", "water"),
    ("combined_liquid_util_pct", "liquid"),
    ("combined_power_util_pct", "power"),
]:
    ax.plot(host_decline.year, host_decline[column], label=label)
ax.plot(host_decline.year, 100.0 * host_decline.requested_util, color="black",
        linewidth=2.5, label="NeqSim requested maximum")
ax.axhline(100.0, color="red", linestyle="--")
ax.set(xlabel="Year", ylabel="Requested utilization [%]",
       title="Declining Host A: annual capacity dimensions")
ax.legend(ncol=3)
plt.show()

## Brownfield modification catalogue

Modifications target the physical constraint and include shutdown/integration effects. Costs below are synthetic screening placeholders.

In [ ]:
brownfield_modifications = pd.DataFrame([
    ["SURF loop / second riser", "shared SURF pressure/capacity", 250, 1.20, 45, "subsea installation and host riser slot"],
    ["Inlet separator internals/rerate", "gas/liquid inlet", 80, 1.15, 20, "verification of vessel/nozzles/internals"],
    ["Parallel separation train", "liquid/gas separation", 300, 1.35, 60, "weight, space, flare and shutdown"],
    ["Water-treatment train", "produced water", 140, 1.40, 30, "discharge quality and disposal"],
    ["Compression rerate/new wheel", "gas compression", 180, 1.18, 35, "driver, antisurge and cooler"],
    ["New compressor train", "gas/export pressure", 450, 1.45, 75, "power, weight, layout and outage"],
    ["Power import upgrade", "electrical power", 220, 1.30, 40, "cable, switchgear and shutdown"],
    ["Control/measurement tie-in", "operability and allocation", 45, 1.05, 15, "metering and ownership interfaces"],
], columns=[
    "modification", "target_constraint", "capex_musd", "capacity_multiplier",
    "planned_outage_days", "key_integration_risk"
])
display(brownfield_modifications)

## Modification sequencing

Debottlenecking should follow the constraint chain: increasing one unit can expose the next. Rerun the complete host ProcessModel after each candidate and combination.

In [ ]:
modification_sequence = pd.DataFrame([
    [1, "Validate actual host/SURF constraints", "no investment case before model/data QA"],
    [2, "Operational optimization and allocation", "lowest-cost capacity recovery"],
    [3, "Internals/driver rerate", "use installed equipment margin"],
    [4, "Water/compression targeted modification", "remove controlling dimension"],
    [5, "Parallel train or SURF loop", "large structural expansion"],
    [6, "Alternative host or standalone", "fallback if integration is unattractive"],
], columns=["priority", "action", "decision_logic"])
display(modification_sequence)

## Standalone versus tieback: typical advantages

Standalone provides reservoir/process optimization, dedicated availability, product control, gas-injection flexibility, clear ownership and less dependence on host cessation. Tieback can reduce CAPEX and schedule, reuse power/export/utilities, increase host utilization and extend host life.

In [ ]:
advantages = pd.DataFrame([
    ["Standalone", "Subsurface optimization", "facility and injection designed around Aurora"],
    ["Standalone", "Capacity ownership", "no competition with base-field production"],
    ["Standalone", "Product control", "dedicated oil/gas/water treatment"],
    ["Standalone", "Late-life autonomy", "not tied to host cessation timing"],
    ["Tieback", "Lower initial CAPEX", "reuse processing, utilities and export"],
    ["Tieback", "Schedule potential", "smaller offshore scope if host ready"],
    ["Tieback", "Area value", "fills declining host and may extend life"],
    ["Tieback", "Lower new footprint", "less new topsides/infrastructure"],
], columns=["route", "advantage", "engineering_reason"])
display(advantages)

## Standalone versus tieback: typical disadvantages

Standalone carries high CAPEX, execution complexity, new export requirements and potential emissions/weight exposure. Tieback carries hydraulic distance, host competition, modifications, shutdown interfaces, product compatibility, metering/tariffs, ownership complexity and decommissioning dependence.

In [ ]:
disadvantages = pd.DataFrame([
    ["Standalone", "High CAPEX", "new hull/platform, process, utilities and export"],
    ["Standalone", "Execution/schedule", "large fabrication and integration scope"],
    ["Standalone", "Utilization risk", "oversized facility after plateau"],
    ["Standalone", "Export access", "new oil/gas logistics and bookings"],
    ["Tieback", "SURF pressure/thermal limits", "distance can constrain wells and restart"],
    ["Tieback", "Capacity competition", "host oil/gas/water/power takes priority"],
    ["Tieback", "Brownfield modifications", "space, weight, shutdown and safety interfaces"],
    ["Tieback", "Product compatibility", "fluid can violate host/export/discharge basis"],
    ["Tieback", "Commercial complexity", "tariffs, allocation, metering and ownership"],
    ["Tieback", "Cessation dependency", "satellite life may exceed host life"],
], columns=["route", "disadvantage", "engineering_reason"])
display(disadvantages)

## Engineering option scorecard

The scorecard is qualitative and keeps the discussion auditable. Scores are not a substitute for gate evidence.

In [ ]:
engineering_scorecard = pd.DataFrame([
    ["Standalone FPSO", 5, 2, 5, 5, 3, 5, 2],
    ["Host A priority", 3, 5, 2, 3, 4, 2, 4],
    ["Host A managed", 3, 5, 3, 2, 4, 2, 3],
    ["Host B remote", 2, 4, 2, 2, 2, 2, 3],
], columns=[
    "option", "subsurface_flexibility", "capex_efficiency", "capacity_control",
    "commercial_simplicity", "schedule", "late_life_autonomy", "area_utilization"
])
engineering_scorecard["unweighted_total"] = engineering_scorecard.iloc[:, 1:].sum(axis=1)
display(engineering_scorecard.sort_values("unweighted_total", ascending=False))

## Detailed engineering conclusion

The six-producer/three-injector reference is plausible for the selected plateau only under the assumed PI and pressure margin. Tubing and flowline diameter are route-dependent; the 65 km option needs substantially more hydraulic assurance than the 12/35 km cases. Two production risers and spare manifold slots provide flexibility but must be justified against cost and reliability.

The greenfield ProcessSystem can be auto-sized and its mechanical constraints exported as JSON. The brownfield case must instead load actual installed equipment. Host decline creates oil/gas ullage, but water, liquid, power or a named unit can remain controlling. Modification benefits must be proven by rerunning the full process and lifecycle with outage and CAPEX.

In [ ]:
deep_engineering_checks = pd.DataFrame([
    ["production well drawdown calculated", not well_drawdown.empty],
    ["tubing alternatives calculated", tubing_screen.wellhead_bara.notna().any()],
    ["flowline alternatives calculated", multiphase_screen.outlet_bara.notna().any()],
    ["gas pipeline alternatives calculated", gas_lateral_screen.outlet_bara.notna().any()],
    ["SURF cost alternatives calculated", len(surf_cost_sensitivity) >= 9],
    ["greenfield design report extracted", len(design_equipment) > 0],
    ["mechanical constraints extracted", not detailed_constraints.empty],
    ["declining host ullage calculated", len(host_decline) > 0],
    ["oil/gas/water/liquid/power checked", all(
        column in host_decline for column in [
            "combined_oil_util_pct", "combined_gas_util_pct",
            "combined_water_util_pct", "combined_liquid_util_pct",
            "combined_power_util_pct"
        ]
    )],
], columns=["engineering_gate", "passed"])
display(deep_engineering_checks)
if not deep_engineering_checks.passed.all():
    raise AssertionError("One or more detailed engineering gates failed.")

## 13. Validation gates

Validation is layered: notebook integrity, PVT/process execution, conservation/schedule sanity, capacity consistency, economics reconstruction and route provenance.

In [ ]:
validation_results = []

def check(name, condition, detail):
    validation_results.append({"check": name, "passed": bool(condition), "detail": detail})

check("portfolio option count", len(option_results) >= 4, len(option_results))
check("standalone annual results", len(standalone_annual) > 0, len(standalone_annual))
check("positive oil", standalone.getCumulativeOilSm3() > 0.0, standalone.getCumulativeOilSm3())
check("pressure declines", standalone.getFinalReservoirPressureBara() < standalone.getInitialReservoirPressureBara(),
      f"{standalone.getInitialReservoirPressureBara():.1f}->{standalone.getFinalReservoirPressureBara():.1f} bara")
check("requested >= admitted peak",
      standalone.getPeakUnconstrainedFacilityUtilization() >= standalone.getPeakFacilityUtilization(),
      "capacity-allocation ordering")
check("facility trains", design.getRequiredParallelTrainCount() >= 1, design.getRequiredParallelTrainCount())
check("economics reconstruction", abs(reconstructed.getNpv() - standalone.getNpvMusd()) < 1e-6,
      npv_check["difference_musd"])
check("geospatial CRS", str(aurora_point.crs).upper().endswith("4326"), aurora_point.crs)
validation_df = pd.DataFrame(validation_results)
display(validation_df)

## Fail loudly on critical errors

The notebook should stop if any critical integrated check fails. Source availability checks remain warnings because public web services can be temporarily unavailable.

In [ ]:
failed = validation_df.loc[~validation_df.passed]
if not failed.empty:
    raise AssertionError("Critical validation failures:\n" + failed.to_string(index=False))
print("All critical integrated validation gates passed.")

## Model limitation register

Limitations are part of the result, not a footnote. They define what decision the notebook can and cannot support.

In [ ]:
limitations = pd.DataFrame([
    ["Reservoir", "tank model and synthetic inventories", "no 3D sweep/conformance"],
    ["Wells", "aggregate PI/tubing", "no well-by-well schedule or lift design"],
    ["SURF", "steady-state screening", "no transient hydrate/slug/restart design"],
    ["Facility", "detailed reference process", "not a complete project simulator"],
    ["Hosts", "synthetic production/capacity", "no statement about real asset ullage"],
    ["Maps", "public screening geometry", "not survey/navigation data"],
    ["Gas transport", "public system context + synthetic booking", "no secured capacity"],
    ["Economics", "synthetic costs/prices/tax basis", "not an investment forecast"],
    ["Uncertainty", "conditional post-processing", "not fully coupled probabilistic model"],
], columns=["domain", "limitation", "decision_consequence"])
display(limitations)

## Recommended next engineering work

1. Reconcile appraisal volumetrics and executable reservoir inventory.

2. Regress PVT to representative laboratory data and add brine chemistry.

3. Connect a well-by-well reservoir/network schedule.

4. Replace route assumptions with surveyed GIS/bathymetry and flow-assurance cases.

5. Import the actual host ProcessModel and existing SURF topology with live mechanical limits.

6. Obtain operator-approved host production, shutdown and modification cases.

7. Configure actual oil/gas/water specifications and Gassco route/booking/tariff terms.

8. Run coupled low/base/high and probabilistic cases, including schedule and modifications.

## Educational conclusion

The notebook demonstrates that a tieback can outperform a standalone facility economically while simultaneously creating more deferment, host dependence and modification risk. It also shows why requested utilization, product compliance and route topology must accompany NPV.

The exact ranking is a synthetic teaching result. It says nothing about a named NCS field or facility.

In [ ]:
final_snapshot = portfolio_summary.sort_values("npv_musd", ascending=False)[[
    "option", "eligible", "npv_musd", "breakeven_oil_usd_bbl",
    "oil_msm3", "deferred_oil_msm3", "peak_requested_util_pct", "off_spec_years"
]]
display(final_snapshot)
print("Notebook cells are designed for education about field and area development only.")

## Suggested student exercises

1. Select a public SODIR discovery for context but preserve a strict public/synthetic ledger.

2. Move Aurora Sør and rerun host-distance and route-topology screening.

3. Add a seabed-cost raster or bathymetry-aware least-cost path.

4. Replace the gas lateral with several NeqSim flow/diameter/compressor cases.

5. Add dehydration and hydrocarbon-dew-point control to the greenfield process.

6. Import a real ProcessSystem and identify area::equipment bottlenecks.

7. Model a staged host modification with outage, CAPEX and post-modification capacity.

8. Add firm versus interruptible gas transport scenarios and route-specific tariffs.

9. Couple low/base/high reservoir schedules to independent mutable lifecycle models.

10. Build a decision report that separates gates, value, uncertainty and evidence maturity.

## Final disclaimer

**This entire notebook is for education about field and area development.** Do not quote its reservoir, production, facility, host, route, cost, tariff, emissions, NPV or break-even values as facts about a Norwegian asset. Real decisions require operator-validated models, engineering assurance, commercial agreements, regulatory requirements and independent review.